In [ ]:
# ============================================================
# CELDA 0  — PANEL DE CONTROL (único lugar que se edita)6
# ============================================================
from pathlib import Path

VERSION      = 'V8'                      
PROJECT_ROOT = Path(r'C:\Users\rodri\Documents\PINN\b2_proyecto')

# Etiquetas descriptivas (solo informativas, para títulos/JSON)
DATASET_LABEL = 'v6 CORREGIDO (alpha T0,p)'
ARCH_LABEL    = 'PINNModal_v4b'
LOSS_LABEL    = 'compute_loss_v16'
CURR_LABEL    = 'curriculum v20'

# ── Rutas derivadas automáticamente de VERSION ───────────────
VTAG          = VERSION.lower()                      
DATA_DIR      = PROJECT_ROOT / 'dataset' / 'opensees'
TRIALS_ROOT   = PROJECT_ROOT / 'models' / 'trials_31'
TRIALS_DIR    = TRIALS_ROOT / VERSION                 
SCATTER_DIR   = TRIALS_DIR / 'scatter_plots'
DEFINITIVO_DIR = TRIALS_DIR / 'DEFINITIVO_CORREGIDO'

# Alias de compatibilidad: las celdas que aún usan TRIALS_DIR_V8 siguen funcionando
TRIALS_DIR_VER = TRIALS_DIR
TRIALS_DIR_V8  = TRIALS_DIR    # alias retro-compatible → apunta a la VERSION activa

for _d in (TRIALS_ROOT, TRIALS_DIR, SCATTER_DIR, DEFINITIVO_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Helper para nombres de archivo con sufijo de versión:
#   vfile('all_results', 'json')          -> 'all_results_v8.json'
#   vfile('dashboard_31_trials', 'png')   -> 'dashboard_31_trials_v8.png'
def vfile(stem, ext):
    return f'{stem}_{VTAG}.{ext}'

# Rutas de archivos de resultados (derivadas de VERSION)
RESULTS_JSON = TRIALS_DIR / vfile('all_results', 'json')

print('=' * 60)
print('PANEL DE CONTROL NB5')
print('=' * 60)
print(f'  VERSION        : {VERSION}')
print(f'  PROJECT_ROOT   : {PROJECT_ROOT}')
print(f'  TRIALS_DIR     : {TRIALS_DIR}')
print(f'  SCATTER_DIR    : {SCATTER_DIR}')
print(f'  DEFINITIVO_DIR : {DEFINITIVO_DIR}')
print(f'  RESULTS_JSON   : {RESULTS_JSON.name}')
print(f'  DATASET        : {DATASET_LABEL}')
print(f'  ARQUITECTURA   : {ARCH_LABEL}')
print(f'  LOSS           : {LOSS_LABEL}')
print(f'  CURRICULUM     : {CURR_LABEL}')
print('=' * 60)

PANEL DE CONTROL NB5
  VERSION        : V8
  PROJECT_ROOT   : C:\Users\rodri\Documents\PINN\b2_proyecto
  TRIALS_DIR     : C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8
  SCATTER_DIR    : C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\scatter_plots
  DEFINITIVO_DIR : C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\DEFINITIVO_CORREGIDO
  RESULTS_JSON   : all_results_v8.json
  DATASET        : v6 CORREGIDO (alpha T0,p)
  ARQUITECTURA   : PINNModal_v4b
  LOSS           : compute_loss_v16
  CURRICULUM     : curriculum v20


## 📌 Celda 1 — Imports, dispositivo de cómputo y configuración del experimento

### 🎯 ¿Qué hace esta celda?

Es la **celda de inicialización del notebook**. Prepara todo lo necesario para que el resto del código funcione:

1. **Importa librerías** científicas y de deep learning:
   - `numpy`, `pandas`, `matplotlib` → manejo de datos numéricos y gráficos.
   - `h5py` → lectura del dataset `Familia B` almacenado en formato HDF5.
   - `torch`, `torch.nn`, `torch.nn.functional` → construcción de la **PINN** (Physics-Informed Neural Network).
   - `sklearn` → métricas auxiliares ($R^2$, MAE) y split de datos.

2. **Fija una semilla** $\text{SEED}=2026$ para reproducibilidad estricta:
   $$\text{torch.manual\_seed}(\text{SEED}), \quad \text{np.random.seed}(\text{SEED})$$

3. **Detecta GPU vs CPU** (CUDA si está disponible).

4. **Define rutas** del proyecto y crea el directorio `TRIALS_DIR_V5/` donde se guardarán los resultados de los 31 ensayos.

5. **Define la metodología experimental** — *Repeated Hold-Out con Diseño Factorial 2×2* (Arlot & Celisse, 2010):

| Símbolo | Valor | Significado |
|---|---|---|
| $N_{\text{train}}$ | 1000 | Casos por repetición |
| $N_{\text{val}}^{\text{OOD}}$ | 200 | Validación fuera de distribución |
| $N_{\text{test}}^{\text{OOD}}$ | 200 | Test fuera de distribución |
| $K_{\text{reps}}$ | 4 | Repeticiones por trial |

### 🧪 Diseño Factorial 2×2 — esquema

```
                  ┌─── Pool A (TRAIN) ───┐    ┌──── Pool B (OOD) ────┐
                  │                       │    │                       │
Dominio físico:   │   A1 (1000)   A2 (1000)│    │  B1..B8 (200 c/u)   │
                  └───────────────────────┘    └───────────────────────┘
                                                       │
            ┌──────────────────┬──────────────────┬────┴─────────────┬───────────────┐
            │                  │                  │                  │
        Rep 1               Rep 2              Rep 3              Rep 4
     Train=A1            Train=A1           Train=A2           Train=A2
     Val=B1, Test=B2     Val=B3, Test=B4    Val=B5, Test=B6    Val=B7, Test=B8
            │__________________│__________________│__________________│
                 σ_OOD                 σ_TRAIN              σ_OOD
            (Reps 1-2)            (Reps 1-3)            (Reps 3-4)
```

- **Reps 1 vs 2** → mismo TRAIN, distinto OOD → mide **variabilidad por dominio OOD** ($\sigma_{\text{OOD}}$).
- **Reps 1 vs 3** → distinto TRAIN → mide **variabilidad por selección del entrenamiento** ($\sigma_{\text{train}}$).

### ✅ Lo que se obtiene

- `device`: `cuda` o `cpu` configurado.
- Rutas `PROJECT_ROOT`, `DATA_DIR`, `TRIALS_DIR_V5`.
- Path al HDF5 más reciente del dataset `dataset_familyB_OPENSEES_*.h5`.
- Constantes globales: `N_MAX_PISOS = 18`, `N_MODOS = 18`, `BATCH_SIZE = 64`.
- Verificación de **factibilidad** del pool: Pool A ≥ 2000, Pool B ≥ 1600 casos.

In [4]:
# ============================================================
# CELDA 1 -- IMPORTS + DEVICE + RUTAS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import h5py, json, math, shutil
from pathlib import Path
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Rutas: definidas en la CELDA 0 (panel de control).
# Disponibles aquí: PROJECT_ROOT, DATA_DIR, TRIALS_DIR, TRIALS_DIR_V8 (alias),
#                   SCATTER_DIR, DEFINITIVO_DIR, RESULTS_JSON, vfile().
print(f'TRIALS_DIR ({VERSION}): {TRIALS_DIR}')

# ── HDF5 — siempre el mas reciente ───────────────────────────────────────
_H5_NAME = 'dataset_familyB_OPENSEES_v6_DEFINITIVO.h5'
_BASES = [Path.cwd(), *Path.cwd().parents][:4]
_CAND = [b / _H5_NAME for b in _BASES] + \
        [b / '2_DATASET_Cap4' / 'datasets_h5' / _H5_NAME for b in _BASES]
H5_MODAL = next((p for p in _CAND if p.is_file()), None)
assert H5_MODAL is not None, (
    f'No se encontro {_H5_NAME}. Coloquelo junto a este notebook, en la carpeta padre, '
    'o en 2_DATASET_Cap4/datasets_h5/.')  
print(f'HDF5: {H5_MODAL.name}')

# ── Arquitectura ──────────────────────────────────────────────────────────
N_MAX_PISOS  = 18
N_MODOS      = 18
BATCH_SIZE   = 64

# ── Metodología Repeated Hold-Out con Diseño Factorial 2×2 ───────────────
N_TRAIN    = 1000
N_VAL_OOD  = 200
N_TEST_OOD = 200
K_REPS     = 4

MIN_POOL_A = 2 * N_TRAIN
MIN_POOL_B = K_REPS * (N_VAL_OOD + N_TEST_OOD)

print(f'\nMetodología: Repeated Hold-Out OOD — Diseño Factorial 2×2')
print(f'  N_TRAIN   : {N_TRAIN} casos por repetición')
print(f'  N_VAL_OOD : {N_VAL_OOD} casos por repetición')
print(f'  N_TEST_OOD: {N_TEST_OOD} casos por repetición')
print(f'  K_REPS    : {K_REPS} repeticiones por trial')
print(f'  Pool A mínimo: {MIN_POOL_A} casos')
print(f'  Pool B mínimo: {MIN_POOL_B} casos')
print(f'  Trials: 31')
print(f'\nTRIALS_DIR: {TRIALS_DIR}')
print('CELDA 1 OK')

Device: cuda
GPU:  NVIDIA GeForce RTX 3070 Ti
VRAM: 8.6 GB
TRIALS_DIR (V8): C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8
HDF5: dataset_familyB_OPENSEES_v6_DEFINITIVO.h5

Metodología: Repeated Hold-Out OOD — Diseño Factorial 2×2
  N_TRAIN   : 1000 casos por repetición
  N_VAL_OOD : 200 casos por repetición
  N_TEST_OOD: 200 casos por repetición
  K_REPS    : 4 repeticiones por trial
  Pool A mínimo: 2000 casos
  Pool B mínimo: 1600 casos
  Trials: 31

TRIALS_DIR: C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8
CELDA 1 OK


## 📌 Celda 2 — Carga del dataset HDF5 OpenSees v4

### 🎯 ¿Qué hace esta celda?

Abre el archivo HDF5 del dataset `Familia B` (generado con OpenSees) y carga **todos los arrays** que el modelo necesita para entrenar y evaluar. El dataset contiene **~20 000 casos paramétricos** de edificios residenciales chilenos según NCh 433.

### 📦 Datos cargados (estructura del HDF5)

| Grupo HDF5 | Variable | Forma | Significado físico |
|---|---|---|---|
| `inputs/` | `X` | $(N, 25)$ | Features de entrada (geometría, materiales, suelo, zona) |
| `inputs/` | `mask_floors` | $(N, 18)$ | Máscara binaria de pisos activos por edificio |
| `inputs/` | `floor_masses` | $(N, 18)$ | Masas concentradas por piso $m_i$ [t] |
| `modal/` | `T_r` | $(N, 18)$ | Periodos modales $T_r$ [s] |
| `modal/` | `Phi_x, Phi_y, Phi_theta` | $(N, 18, 18)$ | Formas modales $\Phi$ en 3 GDL/piso |
| `modal/` | `mass_acc_x, mass_acc_y` | $(N, 18)$ | Masa modal acumulada (NCh 433 §6.3.3) |
| `modal/` | `omega_r` | $(N, 18)$ | Frecuencias angulares $\omega_r = 2\pi/T_r$ [rad/s] |
| `response/` | `Ux_m, Uy_m` | $(N, 18)$ | Desplazamientos máximos por piso [m] |
| `response/` | `drift_x, drift_y` | $(N, 18)$ | Derivas de entrepiso [adim] |
| `response/` | `Vb_x_kN, Vb_y_kN` | $(N,)$ | Cortante basal [kN] |
| `stiffness/` | `K_global, M_global` | $(N, 54, 54)$ | Matrices $\mathbf{K}, \mathbf{M}$ globales (18 pisos × 3 GDL = 54) |

### 🔧 Normalización de las formas modales $\Phi$

Las formas modales se renormalizan a **norma euclidiana unitaria por modo**:

$$
\hat{\Phi}_r = \frac{\Phi_r}{\|\Phi_r\|_2}, \qquad \|\Phi_r\|_2 = \sqrt{\sum_{i=1}^{N_p} \Phi_{r,i}^2}
$$

**Justificación ingenieril:**
- Las formas modales en dinámica son **invariantes a escala** (se pueden escalar arbitrariamente).
- Normalizar a $\|\Phi_r\| = 1$ hace que la escala sea independiente del número de pisos $N_p$.
- Esto es consistente con la base sinusoidal que aprende la PINN, que genera vectores con norma $\propto \sqrt{N_p}$.
- Sin esta normalización, los gradientes de la **MAC loss** ($L_\phi$) están sesgados por el tamaño del edificio.

### 📐 Matriz global $\mathbf{K}$ — esquema

```
          ┌──────────────────────────────────┐
          │  K_global  (54 × 54)             │
          │                                  │  Piso 1: GDL [Ux, Uy, θz]
          │  ┌─────┐                         │  Piso 2: GDL [Ux, Uy, θz]
          │  │ K_1 │                         │  ...
          │  └─────┘ ← acoplamiento piso-piso│  Piso 18: GDL [Ux, Uy, θz]
          │           ┌─────┐                │
          │           │ K_2 │                │  Total: 18 pisos × 3 GDL = 54
          │           └─────┘                │
          │                  ...             │
          └──────────────────────────────────┘
```

### ✅ Lo que se obtiene

- Todos los tensores numpy listos en RAM.
- `N_CASES`: número total de casos disponibles.
- `N_MODOS_PHYS = 18`: número de modos físicos extraídos.
- Verificación de que **no hay NaN** en las features.
- Estimación de memoria de las matrices $\mathbf{K}, \mathbf{M}$ por batch.

In [5]:
# CELDA 2 -- CARGA HDF5 OPENSEES v4
with h5py.File(H5_MODAL, 'r') as h5:
    X_raw        = h5['inputs/X'][:].astype(np.float32)
    input_feats  = [s.decode() for s in h5['inputs/input_features'][:]]
    mask_floors  = h5['inputs/mask_floors'][:].astype(np.float32)
    floor_masses = h5['inputs/floor_masses'][:].astype(np.float32)
    T_r          = h5['modal/T_r'][:].astype(np.float32)
    Phi_x        = h5['modal/Phi_x'][:].astype(np.float32)
    Phi_y        = h5['modal/Phi_y'][:].astype(np.float32)
    Phi_theta    = h5['modal/Phi_theta'][:].astype(np.float32)
    mass_acc_x   = h5['modal/mass_acc_x'][:].astype(np.float32)
    mass_acc_y   = h5['modal/mass_acc_y'][:].astype(np.float32)
    mass_part_x  = h5['modal/mass_part_x'][:].astype(np.float32)
    mass_part_y  = h5['modal/mass_part_y'][:].astype(np.float32)
    omega_phys   = h5['modal/omega_r'][:].astype(np.float32)
    Ux_m         = h5['response/Ux_m'][:].astype(np.float32)
    Uy_m         = h5['response/Uy_m'][:].astype(np.float32)
    drift_x      = h5['response/drift_x'][:].astype(np.float32)
    drift_y      = h5['response/drift_y'][:].astype(np.float32)
    Vb_x         = h5['response/Vb_x_kN'][:].astype(np.float32)
    Vb_y         = h5['response/Vb_y_kN'][:].astype(np.float32)
    N_pisos_arr  = h5['scalars/N_pisos'][:].astype(int)
    K_global     = h5['stiffness/K_global'][:].astype(np.float32)
    M_global     = h5['stiffness/M_global'][:].astype(np.float32)
    I_rot_floors = h5['stiffness/I_rot_floors'][:].astype(np.float32)

N_CASES      = X_raw.shape[0]
N_MODOS_PHYS = T_r.shape[1]  # 18 slots

# ── Normalizar Phi a norma vectorial=1 por modo ───────────────────────────
# Norma L2 del vector por caso — independiente de N_pisos
# Garantiza escala consistente con la base sinusoidal del PINN
# que genera vectores de norma proporcional a sqrt(N_pisos)
print("Normalizando Phi a norma vectorial=1 por modo...")
for r in range(Phi_x.shape[2]):
    for Phi_arr in [Phi_x, Phi_y, Phi_theta]:
        norma_r = np.sqrt((Phi_arr[:, :, r]**2).sum(axis=1, keepdims=True))
        Phi_arr[:, :, r] = np.where(norma_r > 1e-10,
                                     Phi_arr[:, :, r] / norma_r,
                                     0.0)

print(f'Casos        : {N_CASES}')
print(f'T_r          : {T_r.shape}')
print(f'Phi_x        : {Phi_x.shape}')
print(f'K_global     : {K_global.shape}')
print(f'M_global     : {M_global.shape}')
print(f'omega_phys   : {omega_phys.shape}')
print(f'NaN en X     : {np.isnan(X_raw).sum()}')
print(f'N_MODOS_PHYS : {N_MODOS_PHYS}')

kb_per_case = 54 * 54 * 4 / 1024
print(f'\nMemoria K_global por caso : {kb_per_case:.1f} KB')
print(f'Memoria batch K_global (bs=64) : {kb_per_case*64/1024:.1f} MB')

# Verificar norma vectorial — debe ser 1.0
print('\nVerificacion norma vectorial Phi (debe ser 1.0):')
for i in range(3):
    norma_y = np.sqrt((Phi_y[i,:,0]**2).sum())
    norma_t = np.sqrt((Phi_theta[i,:,1]**2).sum())
    norma_x = np.sqrt((Phi_x[i,:,2]**2).sum())
    print(f'  caso {i}: '
          f'norma Phi_y modo 0={norma_y:.4f}  '
          f'norma Phi_t modo 1={norma_t:.4f}  '
          f'norma Phi_x modo 2={norma_x:.4f}')

print('\nFeatures:')
for i, f in enumerate(input_feats):
    print(f'  [{i:02d}] {f}')

print('\nCELDA 2 OK')
# ── Advertencia features constantes ──────────────────────────────────────
_FEATURES_CONSTANTES = []
for i, feat in enumerate(input_feats):
    v = X_raw[:, i]
    if v.std() < 1e-8:
        _FEATURES_CONSTANTES.append((i, feat, float(v[0])))

if _FEATURES_CONSTANTES:
    print(f'\nAVISO — Features constantes en el HDF5 (std=0):')
    for i, feat, val in _FEATURES_CONSTANTES:
        print(f'  [{i:02d}] {feat} = {val:.0f} siempre '
              f'← se excluye de X_clean en celda 3')
    print(f'  Estas features no aportan información al modelo.')
    print(f'  Son parámetros fijos del generador Familia B v6.')
    print(f'  Se mantienen en el HDF5 por trazabilidad del dataset.')
    print(f'  Se excluyen de COLS_BASE en celda 3 → N_INPUTS={23-len(_FEATURES_CONSTANTES)}')

print('\nCELDA 2 OK')

Normalizando Phi a norma vectorial=1 por modo...
Casos        : 10000
T_r          : (10000, 18)
Phi_x        : (10000, 18, 18)
K_global     : (10000, 54, 54)
M_global     : (10000, 54, 54)
omega_phys   : (10000, 18)
NaN en X     : 0
N_MODOS_PHYS : 18

Memoria K_global por caso : 11.4 KB
Memoria batch K_global (bs=64) : 0.7 MB

Verificacion norma vectorial Phi (debe ser 1.0):
  caso 0: norma Phi_y modo 0=1.0000  norma Phi_t modo 1=1.0000  norma Phi_x modo 2=1.0000
  caso 1: norma Phi_y modo 0=1.0000  norma Phi_t modo 1=1.0000  norma Phi_x modo 2=1.0000
  caso 2: norma Phi_y modo 0=1.0000  norma Phi_t modo 1=1.0000  norma Phi_x modo 2=1.0000

Features:
  [00] N_pisos
  [01] n_unid_lado
  [02] mods_por_depto
  [03] activar_B2
  [04] activar_B3
  [05] L_mod_m
  [06] prof_depto_m
  [07] ancho_corredor_m
  [08] L_nucleo_m
  [09] B_nucleo_m
  [10] h_story_m
  [11] fc_MPa
  [12] gk_kN_m2
  [13] t_muro_nucleo_m
  [14] t_muro_borde_m
  [15] t_muro_mid_m
  [16] suelo
  [17] zona

CELDA 2 OK

AVI

## 📌 Celda 3 — Preprocesamiento global y codificación One-Hot

### 🎯 ¿Qué hace esta celda?

Transforma las features de entrada `X_raw` en un formato amigable para la red neuronal:

1. **Codifica variables categóricas** con *one-hot encoding*:
   - `suelo` ∈ {0,1,2,3} → `suelo_A, suelo_B, suelo_C, suelo_D` (tipos de suelo NCh 433 §4.2)
   - `zona` ∈ {1,2,3} → `zona_1, zona_2, zona_3` (zonas sísmicas NCh 433 §4.1)

2. **Construye las 21 columnas finales** del vector de entrada:

| Categoría | Variables |
|---|---|
| **Geometría global** | `N_pisos`, `n_unid_lado`, `activar_B2`,  |
| **Dimensiones módulo** | `L_mod_m`, `prof_depto_m`, `ancho_corredor_m` |
| **Núcleo/muros** | `L_nucleo_m`, `B_nucleo_m`, `h_story_m` |
| **Materiales** | `fc_MPa` (resistencia hormigón), `gk_kN_m2` (carga gravitacional) |
| **Espesores muro** | `t_muro_nucleo_m`, `t_muro_borde_m`, `t_muro_mid_m` |
| **Suelo (OHE)** | `suelo_A`, `suelo_B`, `suelo_C`, `suelo_D` |
| **Zona sísmica (OHE)** | `zona_1`, `zona_2`, `zona_3` |

3. **Transformación logarítmica** de las salidas con respuesta sísmica:

$$
\tilde{y}_i = \begin{cases} \log(y_i) & \text{si } y_i > 10^{-8} \\ 0 & \text{en caso contrario} \end{cases}
$$

Para: $T_r$, $U_x$, $U_y$, $\text{drift}_x$, $\text{drift}_y$, $V_{b,x}$, $V_{b,y}$.

**¿Por qué log?**
- La respuesta sísmica varía sobre **varios órdenes de magnitud** (un edificio de 6 pisos vs uno de 18).
- $V_b$ puede ir desde 100 kN a 100 000 kN → MSE en escala lineal sería dominado por edificios altos.
- Predecir $\log(y)$ corresponde a minimizar el **error relativo**, que es más útil ingenierilmente.

### ✅ Lo que se obtiene

- `X_clean` ∈ $\mathbb{R}^{N \times 23}$: matriz de features lista para la red.
- `N_INPUTS = 21`: dimensión del input.
- Arrays log-transformados: `logT_global, logUx, logUy, logDx, logDy, logVbx, logVby`.

In [6]:
# CELDA 3 -- PREPROCESAMIENTO GLOBAL + ONE-HOT
df_X = pd.DataFrame(X_raw, columns=input_feats)
suelo_map_inv = {0:'A', 1:'B', 2:'C', 3:'D'}
suelo_ohe = pd.get_dummies(
    df_X['suelo'].map(suelo_map_inv), prefix='suelo'
).astype(np.float32)
zona_ohe = pd.get_dummies(
    df_X['zona'].astype(int), prefix='zona'
).astype(np.float32)
df_X = df_X.drop(columns=['suelo', 'zona'])
df_X = pd.concat([df_X, suelo_ohe, zona_ohe], axis=1)
for col in ['suelo_A','suelo_B','suelo_C','suelo_D']:
    if col not in df_X.columns: df_X[col] = 0.0
for col in ['zona_1','zona_2','zona_3']:
    if col not in df_X.columns: df_X[col] = 0.0

COLS_BASE = [
    'N_pisos', 'n_unid_lado',
    # 'mods_por_depto',  ← ELIMINADO: constante=2, std=0
    'activar_B2',
    # 'activar_B3',      ← ELIMINADO: constante=1, std=0
    'L_mod_m', 'prof_depto_m', 'ancho_corredor_m',
    'L_nucleo_m', 'B_nucleo_m', 'h_story_m',
    'fc_MPa', 'gk_kN_m2',
    't_muro_nucleo_m', 't_muro_borde_m', 't_muro_mid_m',
    'suelo_A', 'suelo_B', 'suelo_C', 'suelo_D',
    'zona_1', 'zona_2', 'zona_3',
]

faltantes = [c for c in COLS_BASE if c not in df_X.columns]
if faltantes:
    print(f'AVISO faltantes: {faltantes}')
else:
    df_X = df_X[COLS_BASE]

X_clean  = df_X.to_numpy(dtype=np.float32)
N_INPUTS = X_clean.shape[1]

with np.errstate(divide='ignore', invalid='ignore'):
    logT_global = np.where(T_r > 1e-8, np.log(np.where(T_r > 1e-8, T_r, 1.0)), 0.0).astype(np.float32)
    logUx       = np.where(Ux_m > 1e-8,    np.log(np.where(Ux_m > 1e-8,    Ux_m,    1.0)), 0.0).astype(np.float32)
    logUy       = np.where(Uy_m > 1e-8,    np.log(np.where(Uy_m > 1e-8,    Uy_m,    1.0)), 0.0).astype(np.float32)
    logDx       = np.where(drift_x > 1e-8, np.log(np.where(drift_x > 1e-8, drift_x, 1.0)), 0.0).astype(np.float32)
    logDy       = np.where(drift_y > 1e-8, np.log(np.where(drift_y > 1e-8, drift_y, 1.0)), 0.0).astype(np.float32)
    logVbx      = np.where(Vb_x > 1e-8,    np.log(np.where(Vb_x > 1e-8,    Vb_x,    1.0)), 0.0).astype(np.float32)
    logVby      = np.where(Vb_y > 1e-8,    np.log(np.where(Vb_y > 1e-8,    Vb_y,    1.0)), 0.0).astype(np.float32)

print(f'N_INPUTS (con one-hot, sin constantes): {N_INPUTS}')
print(f'X_clean : {X_clean.shape}')
print('CELDA 3 OK')

N_INPUTS (con one-hot, sin constantes): 21
X_clean : (10000, 21)
CELDA 3 OK


## 📌 Celda 4 — Arquitectura PINN v4 (definitiva)

### 🎯 ¿Qué hace esta celda?

Define la **arquitectura de red neuronal definitiva** `PINNModal_v4` con una innovación clave: **dos encoders independientes** para evitar la *contaminación de gradientes* (Wang et al., 2022).

### 🧠 Motivación experimental

| Versión | Encoder | T1_MAPE | Sesgo |
|---|---|---|---|
| `PINNModal_v2` | Compartido para todo | ~25% | +0.878 ❌ |
| **`PINNModal_v4`** | **encoder_T1 separado** | **7.8%** | **+0.089 ✅** |
| GradientBoosting (baseline) | — | — | ≈0 |

> **Diagnóstico**: La información para predecir $T_1$ ya está en las features; el problema era que los gradientes de $V_b$ y $\text{drift}$ (que son mucho mayores) **dominaban** el encoder compartido y sesgaban la predicción de $T_1$.

### 🏗️ Arquitectura — diagrama

```
                  X ∈ ℝ²³ (input features)
                        │
                ┌───────┴────────┐
                ▼                ▼
       ┌────────────────┐ ┌─────────────────────┐
       │  encoder_T1    │ │  encoder_compartido │
       │  (hidden=128)  │ │  (hidden=256)       │
       │                │ │                     │
       │  Linear→LN→SiLU│ │  Linear→LN→SiLU     │
       │  ResBlock × 2  │ │  ResBlock × 4       │
       └───────┬────────┘ └──────────┬──────────┘
               │                     │
               ▼                     ├─────────┬──────────┬─────────┐
        ┌──────────────┐             ▼         ▼          ▼         ▼
        │  head_T1     │      ┌──────────┐┌───────── ─┐┌─────── ─┐ ┌────── ┐
        │  (1 valor)   │      │head_T_rest││head_Phi  ││head_resp│ │head_Vb│
        └──────┬───────┘      │  (17)    ││  (18×18×3)││(18×4)   │ │ (2)   │
               │              └─────┬────┘└─────┬─────┘└────┬────┘ └──┬── ─┘
               ▼                    ▼          ▼            ▼        ▼
              T₁                  T₂..T₁₈     Φx,Φy,Φθ   Ux,Uy,    Vb_x,
                                                          drift_x,  Vb_y
                                                          drift_y
```

### 🧱 Bloque residual (ResBlock)

$$
\mathrm{ResBlock}(x) = \mathrm{SiLU}\big(x + \mathrm{LN}(W_2 \cdot \mathrm{SiLU}(\mathrm{LN}(W_1 x)))\big)
$$

donde **SiLU** (Sigmoid Linear Unit, también llamada *Swish*) es: $\mathrm{SiLU}(x) = x \cdot \sigma(x)$.

### 🔗 Garantía monótona de periodos: $T_1 > T_2 > \cdots > T_{18}$

Se construye la cadena por **diferencias positivas**:

$$
\log T_r = \log T_1 - \sum_{j=1}^{r-1} \Delta_j, \qquad \Delta_j = \mathrm{softplus}(z_j) > 0
$$

Esto **garantiza por diseño** el orden físico de los periodos modales.

### 📐 Normalización de formas modales

Después del head_Phi, cada modo se normaliza por su norma $L_2$:

$$
\hat{\Phi}_r = \frac{\Phi_r \odot \mathrm{mask}}{\|\Phi_r \odot \mathrm{mask}\|_2 + \epsilon}
$$

para evitar la ambigüedad de escala intrínseca de los autovectores.

### ✅ Lo que se obtiene

- Clase `PINNModal_v4` definida.
- Función `build_model()` que instancia el modelo en `device`.

In [7]:
# CELDA 4 -- ARQUITECTURA PINN v4b
# encoder_T1 separado (hidden=128) + encoder_compartido (hidden=256)
# v4b: dx via ratio — head_resp canal 2 = ratio, dx = ratio × dy

class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.SiLU(),
            nn.Linear(dim, dim), nn.LayerNorm(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.net(x))


class PINNModal_v4(nn.Module):
    def __init__(self, n_inputs, n_modos=18,
                 n_max_pisos=18, hidden=256, hidden_T1=128):
        super().__init__()
        self.n_modos     = n_modos
        self.n_max_pisos = n_max_pisos

        # ── encoder_T1 dedicado ───────────────────────────────────────────
        self.encoder_T1 = nn.Sequential(
            nn.Linear(n_inputs, hidden_T1),
            nn.LayerNorm(hidden_T1),
            nn.SiLU(),
            ResBlock(hidden_T1),
            ResBlock(hidden_T1),
        )
        self.head_T1 = nn.Sequential(
            nn.Linear(hidden_T1, 32),
            nn.SiLU(),
            nn.Linear(32, 1)
        )

        # ── encoder compartido ────────────────────────────────────────────
        self.encoder = nn.Sequential(
            nn.Linear(n_inputs, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            ResBlock(hidden), ResBlock(hidden),
            ResBlock(hidden), ResBlock(hidden),
        )
        self.head_T_rest = nn.Sequential(
            nn.Linear(hidden, 128), nn.SiLU(),
            nn.Linear(128, n_modos - 1)
        )
        self.head_Phi = nn.Sequential(
            nn.Linear(hidden, 512), nn.SiLU(),
            nn.Linear(512, n_modos * n_max_pisos * 3),
        )
        self.head_resp = nn.Sequential(
            nn.Linear(hidden, 256), nn.SiLU(),
            nn.Linear(256, n_max_pisos * 4),
        )
        self.head_Vb = nn.Sequential(
            nn.Linear(hidden, 64), nn.SiLU(),
            nn.Linear(64, 2)
        )

    def forward(self, X, mask):
        B = X.shape[0]

        # T1
        h_T1   = self.encoder_T1(X)
        T1_out = self.head_T1(h_T1)

        # T2-T18
        h        = self.encoder(X)
        T_deltas = F.softplus(self.head_T_rest(h))
        T_parts  = [T1_out]
        for r in range(self.n_modos - 1):
            T_parts.append(T_parts[-1] - T_deltas[:, r:r+1])
        logT_pred = torch.cat(T_parts, dim=1)

        # Formas modales
        Phi_raw  = self.head_Phi(h).view(B, self.n_modos, self.n_max_pisos, 3)
        mask_4d  = mask.unsqueeze(1).unsqueeze(-1)
        Phi_raw  = Phi_raw * mask_4d
        norma    = Phi_raw.norm(dim=2, keepdim=True).clamp(min=1e-8)
        Phi_norm = Phi_raw / norma
        Phi_all  = Phi_norm.permute(0, 2, 1, 3)
        Phi_x_p     = Phi_all[:, :, :, 0]
        Phi_y_p     = Phi_all[:, :, :, 1]
        Phi_theta_p = Phi_all[:, :, :, 2]

        # Respuesta por piso — canal 2 ahora es ratio (dx = ratio × dy)
        resp    = self.head_resp(h).view(B, self.n_max_pisos, 4)
        Ux_p    = resp[:, :, 0] * mask
        Uy_p    = resp[:, :, 1] * mask
        ratio_p = F.softplus(resp[:, :, 2]) * mask   # ← ratio > 0
        dy_p    = resp[:, :, 3] * mask

        # dx = expm1(log1p_ratio) * dy  (BUG3 fix: ratio in log1p space)
        dx_p = torch.expm1(ratio_p.clamp(min=0)) * dy_p

        # Corte basal
        Vb_p = self.head_Vb(h)

        # ← 10 outputs — v4b, dx via ratio
        return (logT_pred, Phi_x_p, Phi_y_p, Phi_theta_p,
                Ux_p, Uy_p, dx_p, dy_p, Vb_p, ratio_p)


def build_model():
    m = PINNModal_v4(
        N_INPUTS, N_MODOS, N_MAX_PISOS,
        hidden=256, hidden_T1=128
    ).to(device)
    n_total  = sum(p.numel() for p in m.parameters() if p.requires_grad)
    n_enc_T1 = sum(p.numel() for p in m.encoder_T1.parameters()) + \
               sum(p.numel() for p in m.head_T1.parameters())
    n_enc_sh = sum(p.numel() for p in m.encoder.parameters())
    print(f'Parámetros totales    : {n_total:,}')
    print(f'  encoder_T1+head_T1  : {n_enc_T1:,}')
    print(f'  encoder_compartido  : {n_enc_sh:,}')
    print(f'  N_INPUTS            : {N_INPUTS}  (sin features constantes)')
    return m

_ = build_model()
print('CELDA 4 OK -- PINNModal_v4b  (10 outputs, dx via ratio)')

Parámetros totales    : 1,377,064
  encoder_T1+head_T1  : 74,305
  encoder_compartido  : 536,576
  N_INPUTS            : 21  (sin features constantes)
CELDA 4 OK -- PINNModal_v4b  (10 outputs, dx via ratio)


In [8]:
# ============================================================
# CELDA PASO 1 — Validacion NCh433SpectrumLayer
# Verifica que la implementacion PyTorch diferenciable
# reproduce exactamente el espectro NCh433 de nch433.py
# ============================================================
import sys
sys.path.insert(0, r'C:\Users\rodri\Documents\PINN\b2_proyecto')
from b2core.nch433_torch import NCh433SpectrumLayer, Sa_nch433_numpy
import torch, numpy as np

layer = NCh433SpectrumLayer().to(device)
layer.eval()

test_cases = [
    (1, 'A',  6, [0.10, 0.20, 0.50, 1.00, 2.00]),
    (2, 'B', 10, [0.15, 0.30, 0.75, 1.50]),
    (3, 'C', 14, [0.40, 0.60, 0.90, 1.20, 2.50]),
    (1, 'D', 18, [0.10, 0.75, 1.00, 1.40, 2.00, 3.00]),
    (3, 'D',  8, [0.30, 0.60, 1.00, 1.80]),
]

print('Zona Suelo  N |     max|dSa|     max|dSd|  Status')
print('-'*55)
all_ok = True
for zone, soil, N, T_list in test_cases:
    T_arr  = np.array(T_list, dtype=np.float32)
    Sa_np  = np.array([Sa_nch433_numpy(t, zone, soil, N) for t in T_arr])
    Sd_np  = Sa_np * T_arr**2 / (4.0 * np.pi**2)
    soil_idx = ['A','B','C','D'].index(soil)
    x_raw = np.zeros((len(T_arr), 21), dtype=np.float32)
    x_raw[:, 0]              = N
    x_raw[:, 14 + soil_idx]  = 1.0
    x_raw[:, 18 + (zone-1)]  = 1.0
    T_diag = torch.diag(torch.tensor(T_arr)).to(device)
    x_pt   = torch.tensor(x_raw).to(device)
    with torch.no_grad():
        Sa_m, Sd_m = layer(T_diag, x_pt)
    Sa_pt = Sa_m.diagonal().cpu().numpy()
    Sd_pt = Sd_m.diagonal().cpu().numpy()
    err_Sa = np.abs(Sa_pt - Sa_np).max()
    err_Sd = np.abs(Sd_pt - Sd_np).max()
    ok = err_Sa < 1e-4 and err_Sd < 1e-6
    if not ok: all_ok = False
    print(f'   {zone}     {soil}  {N:2d} | {err_Sa:12.2e} {err_Sd:12.2e}  {"OK" if ok else "FAIL"}')

print('-'*55)
print(f'Resultado: {"ALL PASSED" if all_ok else "SOME FAILED"}')

# Curva espectral visual
import matplotlib.pyplot as plt
T_plot = np.linspace(0.01, 3.0, 300).astype(np.float32)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax_i, zone in enumerate([1, 2, 3]):
    ax = axes[ax_i]
    for soil, ls in zip(['A','B','C','D'], ['-','--',':','-.']):
        Sa_curve = np.array([Sa_nch433_numpy(t, zone, soil, 12) for t in T_plot])
        ax.plot(T_plot, Sa_curve / 9.80665, ls=ls, label=f'Suelo {soil}')
    ax.set_xlabel('T [s]'); ax.set_ylabel('Sa/g')
    ax.set_title(f'Zona {zone} (N=12)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('Espectro NCh433 — Validacion PyTorch', fontsize=12)
plt.tight_layout(); plt.show()
print('CELDA PASO 1 OK')


Zona Suelo  N |     max|dSa|     max|dSd|  Status
-------------------------------------------------------
   1     A   6 |     1.97e-08     7.85e-10  OK
   2     B  10 |     6.27e-08     5.45e-10  OK
   3     C  14 |     9.97e-08     2.43e-09  OK
   1     D  18 |     1.15e-07     4.49e-09  OK
   3     D   8 |     2.30e-07     1.36e-08  OK
-------------------------------------------------------
Resultado: ALL PASSED
CELDA PASO 1 OK


C:\Users\rodri\AppData\Local\Temp\ipykernel_7804\3856476707.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 📌 Celda 5 — Función de pérdida PINN v16 (definitiva)

### 🎯 ¿Qué hace esta celda?

Define la **pérdida total** del PINN como una **combinación ponderada** de términos data-driven y términos físicos. La versión `v16` separa los términos para garantizar gradientes "limpios" hacia cada encoder e introduce un término de **ratio de derivas**.

> **Cambio clave v15 → v16:** se elimina `dx` (deriva en X) de $\mathcal{L}_{\text{data\_resp}}$ y se supervisa indirectamente mediante un nuevo término $\mathcal{L}_{\text{ratio}}$ que predice el cociente $\text{drift}_x/\text{drift}_y$. Motivo documentado en el código: $\text{drift}_x$ no es predecible en valor absoluto desde las features disponibles, pero el ratio relativo sí es estable.

### 🧮 Pérdida total (8 términos)

$$
\mathcal{L}_{\text{total}} = \lambda_{\text{r}}\,\mathcal{L}_{\text{data\_resp}} + \lambda_{\text{ratio}}\,\mathcal{L}_{\text{ratio}} + \lambda_{\text{T}_r}\,\mathcal{L}_{\text{data\_T\_rest}} + \lambda_{\text{T}_1}\,\mathcal{L}_{\text{data\_T}_1} + \lambda_{T_1}\,\mathcal{L}_{T_1} + \lambda_\phi\,\mathcal{L}_\phi + \lambda_{\text{eig}}\,\mathcal{L}_{\text{eig}} + \lambda_{\text{o}}\,\mathcal{L}_{\text{ortho}}
$$

### 📊 Desglose de los 8 términos

| Término | Qué supervisa | Gradiente fluye a → |
|---|---|---|
| $\mathcal{L}_{\text{data\_resp}}$ | $\mathrm{MSE}$ de $U_x, U_y, \text{drift}_y, V_{b,x}, V_{b,y}$ (**sin** $\text{drift}_x$) | encoder_compartido |
| $\mathcal{L}_{\text{ratio}}$ | $\mathrm{MSE}$ enmascarado de $\widehat{\text{ratio}}$ vs $\text{drift}_x/\text{drift}_y$ | encoder_compartido |
| $\mathcal{L}_{\text{data\_T\_rest}}$ | $\frac{1}{N_v}\sum_{r=2}^{18} \text{mask}_r (\widehat{\log T_r} - \log T_r)^2$ | encoder_compartido |
| $\mathcal{L}_{\text{data\_T}_1}$ | $\mathrm{MSE}(\widehat{\log T_1}, \log T_1)$ | **encoder_T1** |
| $\mathcal{L}_{T_1}$ (explícito) | $\mathrm{MSE}(\widehat{\log T_1}, \log T_1)$ (refuerzo) | encoder_T1 |
| $\mathcal{L}_\phi$ (MAC loss) | $\frac{1}{N_M}\sum_r (1 - \mathrm{MAC}_r)$ | encoder_compartido |
| $\mathcal{L}_{\text{eig}}$ (eigen) | residuo de $\mathbf{K}\Phi = \omega^2\mathbf{M}\Phi$ | encoder_compartido |
| $\mathcal{L}_{\text{ortho}}$ (M-orto) | $\|\Phi^\top \mathbf{M} \Phi - \mathbf{I}\|^2$ | encoder_compartido |

> Los términos $\mathcal{L}_\phi$, $\mathcal{L}_{\text{eig}}$ y $\mathcal{L}_{\text{ortho}}$ solo se computan cuando `stage_active` los activa (ver curriculum); en otro caso valen 0.

### 🔬 Pérdida física $\mathcal{L}_{\text{eig}}$ — el corazón del PINN

Fuerza el **problema generalizado de autovalores** de la dinámica modal, con $\mathbf{K}$ y $\mathbf{M}$ escaladas por $\max|\mathbf{K}|$ del caso para evitar problemas de magnitud entre edificios:

$$
\mathbf{K}\,\Phi_r = \omega_r^2 \,\mathbf{M}\,\Phi_r, \qquad \omega_r = \frac{2\pi}{T_r}
$$

El residuo normalizado, evaluado sobre los primeros $R = \min(N_{\text{modos phys}}, \cdot)$ modos:

$$
\mathcal{L}_{\text{eig}} = \mathbb{E}\!\left[\frac{\|\mathbf{K}\Phi_r - \omega_r^2 \mathbf{M}\Phi_r\|_2^2}{\|\mathbf{K}\Phi_r\|_2^2 + \|\omega_r^2 \mathbf{M}\Phi_r\|_2^2 + \epsilon}\right]
$$

El período se reconstruye desnormalizando la predicción ($T = \exp(\widehat{\log T}\cdot\sigma + \mu)$) y los modos se reensamblan en formato global de 54 GDL intercalando X/Y/θ antes de normalizar.

### 🎯 Pérdida modal MAC ($\mathcal{L}_\phi$)

**Modal Assurance Criterion** (Allemang & Brown, 1982), evaluado por modo según el patrón cíclico de dirección:

$$
\mathrm{MAC}(\Phi^{\text{pred}}_r, \Phi^{\text{true}}_r) = \frac{(\Phi^{\text{pred}\,\top}_r \Phi^{\text{true}}_r)^2}{(\Phi^{\text{pred}\,\top}_r \Phi^{\text{pred}}_r)(\Phi^{\text{true}\,\top}_r \Phi^{\text{true}}_r)}
$$

- $\mathrm{MAC} = 1$: formas modales idénticas (excepto escala).
- $\mathrm{MAC} \geq 0.9$: aceptable (Pastor et al., 2012).

> **Patrón cíclico:** los modos se ordenan alternando dirección — modo $r$ con $r\bmod 3 = 0 \to$ Y, $=1 \to$ T (torsional), $=2 \to$ X.

### 🎼 Pérdida de M-ortogonalidad ($\mathcal{L}_{\text{ortho}}$)

Propiedad clásica de los modos normalizados a masa, penalizada con MSE sobre los primeros $R$ modos:

$$
\Phi^\top \mathbf{M}\, \Phi = \mathbf{I}
$$

### ✅ Lo que se obtiene

- Función `compute_loss_v16(batch, model_out, lambdas, scalers, stage_active)` que devuelve $\mathcal{L}_{\text{total}}$ y un diccionario con cada componente (incluido `ratio`).
- Alias `compute_loss = compute_loss_v16` para compatibilidad con el resto del notebook.
- `model_out` tiene **10 salidas**: `logT, Phi_x, Phi_y, Phi_t, Ux, Uy, dx, dy, Vb, ratio`.

In [9]:
# CELDA 5 -- LOSS PINN v16
# Cambios vs v15:
#   + L_ratio = MSE(ratio_p, drift_x/drift_y) — NUEVO
#   - dx_p eliminado de L_data_resp (supervisado via ratio)
#   Motivo: drift_x no predecible en valor absoluto desde features
#   disponibles. Predecir ratio relativo es más estable.

def compute_loss_v16(batch, model_out, lambdas, scalers, stage_active):
    logT_p, Phi_x_p, Phi_y_p, Phi_t_p, \
        Ux_p, Uy_p, dx_p, dy_p, Vb_p, ratio_p = model_out  # 10 outputs

    logT_true  = batch['logT'].to(device)
    Phi_x_true = batch['Phi_x'].to(device)
    Phi_y_true = batch['Phi_y'].to(device)
    Phi_t_true = batch['Phi_theta'].to(device)
    Ux_true    = batch['logUx'].to(device)
    Uy_true    = batch['logUy'].to(device)
    dy_true    = batch['logDy'].to(device)
    Vbx_true   = batch['logVbx'].to(device)
    Vby_true   = batch['logVby'].to(device)
    ratio_true = batch['ratio_dx_dy'].to(device)
    mask       = batch['mask'].to(device)

    ld = {}

    # ── L_data_resp — Vb, Uy, Ux, dy (dx via ratio) ──────────────────────
    ld['data_resp'] = (
        F.mse_loss(Ux_p * mask, Ux_true * mask) +
        F.mse_loss(Uy_p * mask, Uy_true * mask) +
        F.mse_loss(dy_p * mask, dy_true * mask) +
        F.mse_loss(Vb_p[:, 0], Vbx_true)        +
        F.mse_loss(Vb_p[:, 1], Vby_true)
    )

    # ── L_ratio — ratio drift_x/drift_y ──────────────────────────────────
    n_activos   = mask.sum().clamp(min=1.0)
    ld['ratio'] = (
        (ratio_p - ratio_true).pow(2) * mask
    ).sum() / n_activos

    # ── L_data_T_rest — T2-T18 ───────────────────────────────────────────
    mask_T  = (batch['T_r'].to(device)[:, 1:] > 1e-6).float()
    n_valid = mask_T.sum().clamp(min=1.0)
    ld['data_T_rest'] = (
        (logT_p[:, 1:] - logT_true[:, 1:]).pow(2) * mask_T
    ).sum() / n_valid

    # ── L_data_T1 ────────────────────────────────────────────────────────
    ld['data_T1'] = F.mse_loss(logT_p[:, 0], logT_true[:, 0])

    # ── L_phi ────────────────────────────────────────────────────────────
    L_phi = torch.tensor(0.0, device=device)
    if stage_active.get('phi', False):
        for r in range(N_MODOS):
            ciclo = r % 3
            if ciclo == 0:
                Phi_p_r = Phi_y_p[:, :, r]; Phi_t_r = Phi_y_true[:, :, r]
            elif ciclo == 1:
                Phi_p_r = Phi_t_p[:, :, r]; Phi_t_r = Phi_t_true[:, :, r]
            else:
                Phi_p_r = Phi_x_p[:, :, r]; Phi_t_r = Phi_x_true[:, :, r]
            pp     = Phi_p_r * mask
            pt     = Phi_t_r * mask
            num    = (pp * pt).sum(dim=1) ** 2
            den    = ((pp**2).sum(dim=1) *
                      (pt**2).sum(dim=1)).clamp(min=1e-12)
            valido = (pt.pow(2).sum(dim=1) > 1e-10).float()
            n_val  = valido.sum().clamp(min=1.0)
            L_phi  = L_phi + (valido * (1.0 - num/den)).sum() / n_val
        L_phi = L_phi / N_MODOS
    ld['phi'] = L_phi

    # ── L_eig ────────────────────────────────────────────────────────────
    L_eig = torch.tensor(0.0, device=device)
    if stage_active.get('eig', False):
        K_b = batch['K_global'].to(device)
        M_b = batch['M_global'].to(device)
        _Bs = Phi_x_p.shape[0]
        _mf = batch['mask'].to(device).unsqueeze(-1).expand(-1,-1,3).reshape(_Bs,54)
        _m2 = _mf.unsqueeze(2) * _mf.unsqueeze(1)
        K_b = K_b * _m2;  M_b = M_b * _m2
        K_scale = K_b.abs().amax(dim=(-2,-1)).clamp(min=1.0)
        K_b = K_b / K_scale.unsqueeze(-1).unsqueeze(-1)
        M_b = M_b / K_scale.unsqueeze(-1).unsqueeze(-1)
        R   = min(N_MODOS_PHYS, Phi_x_p.shape[2])
        Phi_g = torch.zeros(Phi_x_p.shape[0], 54, R,
                            dtype=Phi_x_p.dtype, device=device)
        Phi_g[:, 0::3, :] = Phi_x_p[:, :, :R]
        Phi_g[:, 1::3, :] = Phi_y_p[:, :, :R]
        Phi_g[:, 2::3, :] = Phi_t_p[:, :, :R]
        Phi_g   = Phi_g / Phi_g.norm(dim=1, keepdim=True).clamp(min=1e-8)
        logT_std_R  = torch.tensor(scalers['logT_std'][:R],
                                   dtype=torch.float32, device=device)
        logT_mean_R = torch.tensor(scalers['logT_mean'][:R],
                                   dtype=torch.float32, device=device)
        T_pred  = torch.exp(
            logT_p[:, :R] * logT_std_R + logT_mean_R).clamp(min=1e-6)
        omega2  = (2.0 * math.pi / T_pred) ** 2
        KPhi    = torch.bmm(K_b, Phi_g)
        MPhi    = torch.bmm(M_b, Phi_g)
        o2MPhi  = omega2.unsqueeze(1) * MPhi
        residual= KPhi - o2MPhi
        denom   = (KPhi.pow(2).sum(dim=1) +
                   o2MPhi.pow(2).sum(dim=1) + 1e-12)
        L_eig   = (residual.pow(2).sum(dim=1) / denom).mean()
    ld['eig'] = L_eig

    # ── L_ortho ───────────────────────────────────────────────────────────
    L_ortho = torch.tensor(0.0, device=device)
    if stage_active.get('ortho', False):
        M_b = batch['M_global'].to(device)
        _Bs = Phi_x_p.shape[0]
        _mf = batch['mask'].to(device).unsqueeze(-1).expand(-1,-1,3).reshape(_Bs,54)
        _m2 = _mf.unsqueeze(2) * _mf.unsqueeze(1)
        M_b = M_b * _m2
        K_s = M_b.abs().amax(dim=(-2,-1)).clamp(min=1.0)
        M_b = M_b / K_s.unsqueeze(-1).unsqueeze(-1)
        R   = min(N_MODOS_PHYS, Phi_x_p.shape[2])
        Phi_g = torch.zeros(Phi_x_p.shape[0], 54, R,
                            dtype=Phi_x_p.dtype, device=device)
        Phi_g[:, 0::3, :] = Phi_x_p[:, :, :R]
        Phi_g[:, 1::3, :] = Phi_y_p[:, :, :R]
        Phi_g[:, 2::3, :] = Phi_t_p[:, :, :R]
        Phi_g   = Phi_g / Phi_g.norm(dim=1, keepdim=True).clamp(min=1e-8)
        MPhi    = torch.bmm(M_b, Phi_g)
        G       = torch.bmm(Phi_g.transpose(1, 2), MPhi)
        I_mat   = torch.eye(R, device=device).unsqueeze(0)
        L_ortho = ((G - I_mat) ** 2).mean()
    ld['ortho'] = L_ortho

    # ── L_total ───────────────────────────────────────────────────────────
    L_total = (
        lambdas.get('data_resp',   0.0) * ld['data_resp']   +
        lambdas.get('ratio',       0.0) * ld['ratio']       +
        lambdas.get('data_T_rest', 0.0) * ld['data_T_rest'] +
        lambdas.get('data_T1',     0.0) * ld['data_T1']     +
        lambdas.get('phi',         0.0) * ld['phi']         +
        lambdas.get('eig',         0.0) * ld['eig']         +
        lambdas.get('ortho',       0.0) * ld['ortho']
    )
    return L_total, {
        k: v.item() if hasattr(v, 'item') else float(v)
        for k, v in ld.items()
    }

compute_loss = compute_loss_v16

print('CELDA 5 OK -- Loss v16')
print('  L_data_resp  : MSE(Ux, Uy, dy, Vb)  ← dx via ratio')
print('  L_ratio      : MSE(ratio_p, drift_x/drift_y)')
print('  L_data_T_rest: MSE(logT[:,1:])')
print('  L_data_T1    : MSE(logT[:,0])')
print('  L_T1         : término explícito T1')
print('  L_phi        : MAC loss')
print('  L_eig        : K·Phi = w²·M·Phi')
print('  L_ortho      : Phi^T·M·Phi = I')

CELDA 5 OK -- Loss v16
  L_data_resp  : MSE(Ux, Uy, dy, Vb)  ← dx via ratio
  L_ratio      : MSE(ratio_p, drift_x/drift_y)
  L_data_T_rest: MSE(logT[:,1:])
  L_data_T1    : MSE(logT[:,0])
  L_T1         : término explícito T1
  L_phi        : MAC loss
  L_eig        : K·Phi = w²·M·Phi
  L_ortho      : Phi^T·M·Phi = I


## 📌 Celda 6 — Dataset PyTorch + funciones auxiliares

### 🎯 ¿Qué hace esta celda?

Define las **abstracciones PyTorch** necesarias para iterar sobre el dataset durante el entrenamiento.

### 🔧 Componentes definidos

#### 1. Clase `ModalDataset(Dataset)`

Hereda de `torch.utils.data.Dataset`. Para cada caso $j$, devuelve un diccionario con **todos los tensores** del problema (input normalizado, máscara, masas, periodos log-normalizados, formas modales, respuestas, $\mathbf{K}$, $\mathbf{M}$, etc.).

La normalización de X se aplica al vuelo:

$$
\tilde{x}_j = \frac{x_j - \mu_X^{\text{train}}}{\sigma_X^{\text{train}}}
$$

donde $\mu_X^{\text{train}}, \sigma_X^{\text{train}}$ se calculan **solo** sobre el conjunto de entrenamiento (para evitar **data leakage**).

#### 2. Función `compute_scalers(idx_train)`

Calcula los **estadísticos de normalización** ($\mu, \sigma$) usando *solo* los índices de entrenamiento. Para los periodos log se normaliza modo a modo:

$$
\widetilde{\log T_r} = \frac{\log T_r - \mu_{\log T_r}^{\text{train}}}{\sigma_{\log T_r}^{\text{train}}}
$$

Análogamente para $U_x, U_y, \text{drift}_x, \text{drift}_y, V_{b,x}, V_{b,y}$ (pero sobre el **máximo por caso**, no por piso).

#### 3. Función `make_dataloaders(idx_tr, idx_val, idx_test, scalers)`

Crea tres `DataLoader`s. El de entrenamiento usa **`WeightedRandomSampler`** para balancear el número de pisos:

$$
w_j = \frac{1}{\text{counts}[N_{\text{pisos},j}]}
$$

> **¿Por qué?** Si el dataset tiene muchos más edificios de 8 pisos que de 18, sin balanceo la red **sobreajusta** a los edificios bajos. El sampler ponderado fuerza una distribución uniforme sobre $N_{\text{pisos}}$.

#### 4. Función `build_model()`

Construye una instancia fresca de `PINNModal_v4` con los hiperparámetros definitivos:

$$
\text{hidden}=256, \quad \text{hidden}_{T_1}=128, \quad N_{\text{modos}}=18, \quad N_{\text{max pisos}}=18
$$

### ✅ Lo que se obtiene

- `ModalDataset` — generador de batches.
- `compute_scalers(idx)` — calcula $\mu, \sigma$ por trial.
- `make_dataloaders(...)` — crea los 3 DataLoaders.
- `build_model()` — alias para construir el modelo.

In [10]:
# CELDA 6 -- DATASET PYTORCH + FUNCIONES AUXILIARES

class ModalDataset(Dataset):
    def __init__(self, indices, scalers):
        self.idx = indices
        self.sc  = scalers
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]; sc = self.sc
        return {
            'X':           torch.tensor((X_clean[j]-sc['X_mean'])/sc['X_std'], dtype=torch.float32),
            'mask':        torch.tensor(mask_floors[j],    dtype=torch.float32),
            'masses':      torch.tensor(floor_masses[j],   dtype=torch.float32),
            'N_pisos':     torch.tensor(N_pisos_arr[j],    dtype=torch.long),
            'h_story':     torch.tensor(X_clean[j, COLS_BASE.index('h_story_m')], dtype=torch.float32),
            'logT':        torch.tensor(sc['logT_norm'][j],   dtype=torch.float32),
            'T_r':         torch.tensor(T_r[j],               dtype=torch.float32),
            'Phi_x':       torch.tensor(Phi_x[j],             dtype=torch.float32),
            'Phi_y':       torch.tensor(Phi_y[j],             dtype=torch.float32),
            'Phi_theta':   torch.tensor(Phi_theta[j],         dtype=torch.float32),
            'mass_acc_x':  torch.tensor(mass_acc_x[j],        dtype=torch.float32),
            'mass_acc_y':  torch.tensor(mass_acc_y[j],        dtype=torch.float32),
            'logUx':       torch.tensor(sc['logUx_norm'][j],  dtype=torch.float32),
            'logUy':       torch.tensor(sc['logUy_norm'][j],  dtype=torch.float32),
            'logDx':       torch.tensor(sc['logDx_norm'][j],  dtype=torch.float32),
            'logDy':       torch.tensor(sc['logDy_norm'][j],  dtype=torch.float32),
            'logVbx':      torch.tensor(sc['logVbx_norm'][j], dtype=torch.float32),
            'logVby':      torch.tensor(sc['logVby_norm'][j], dtype=torch.float32),
            'Ux_raw':      torch.tensor(Ux_m[j],              dtype=torch.float32),
            'Uy_raw':      torch.tensor(Uy_m[j],              dtype=torch.float32),
            'Vbx_raw':     torch.tensor(Vb_x[j],             dtype=torch.float32),
            'Vby_raw':     torch.tensor(Vb_y[j],             dtype=torch.float32),
            'drift_x_raw': torch.tensor(drift_x[j],          dtype=torch.float32),
            'drift_y_raw': torch.tensor(drift_y[j],          dtype=torch.float32),
            'ratio_dx_dy': torch.tensor(
                np.log1p(drift_x[j] / np.clip(drift_y[j], 1e-6, None)),
                dtype=torch.float32),
            'omega_phys':  torch.tensor(omega_phys[j],        dtype=torch.float32),
            'K_global':    torch.tensor(K_global[j],          dtype=torch.float32),
            'M_global':    torch.tensor(M_global[j],          dtype=torch.float32),
        }


def compute_scalers(idx_train):
    X_m = X_clean[idx_train].mean(axis=0)
    X_s = X_clean[idx_train].std(axis=0); X_s[X_s < 1e-8] = 1.0

    N_MODOS_LOG = logT_global.shape[1]
    logT_m = np.zeros(N_MODOS_LOG, dtype=np.float32)
    logT_s = np.ones(N_MODOS_LOG,  dtype=np.float32)
    for r in range(N_MODOS_LOG):
        vals = logT_global[idx_train, r]; vals = vals[vals != 0]
        if len(vals) > 0:
            logT_m[r] = vals.mean()
            logT_s[r] = max(vals.std(), 1e-8)

    def scaler_pisos(arr_2d):
        """Media/std sobre todos los pisos activos — corrección bug .max(axis=1)"""
        vals = []
        for i in idx_train:
            n = int(N_pisos_arr[i])
            v = arr_2d[i, :n].flatten()
            v = v[v != 0]
            if len(v) > 0:
                vals.extend(v.tolist())
        vals = np.array(vals, dtype=np.float32)
        if len(vals) == 0:
            return 0.0, 1.0
        return float(vals.mean()), max(float(vals.std()), 1e-8)

    def scaler_1d(arr):
        v = arr[idx_train]; v = v[v != 0]
        if len(v) == 0: return 0.0, 1.0
        return float(v.mean()), max(float(v.std()), 1e-8)

    Ux_m_s,  Ux_s_s  = scaler_pisos(logUx)
    Uy_m_s,  Uy_s_s  = scaler_pisos(logUy)
    Dx_m_s,  Dx_s_s  = scaler_pisos(logDx)
    Dy_m_s,  Dy_s_s  = scaler_pisos(logDy)
    Vbx_m_s, Vbx_s_s = scaler_1d(logVbx)
    Vby_m_s, Vby_s_s = scaler_1d(logVby)

    logT_norm   = (logT_global  - logT_m) / logT_s
    logUx_norm  = (logUx - Ux_m_s)  / Ux_s_s
    logUy_norm  = (logUy - Uy_m_s)  / Uy_s_s
    logDx_norm  = (logDx - Dx_m_s)  / Dx_s_s
    logDy_norm  = (logDy - Dy_m_s)  / Dy_s_s
    logVbx_norm = (logVbx - Vbx_m_s) / Vbx_s_s
    logVby_norm = (logVby - Vby_m_s) / Vby_s_s

    return {
        'X_mean': X_m, 'X_std': X_s,
        'logT_mean': logT_m, 'logT_std': logT_s, 'logT_norm': logT_norm,
        'logUx_norm': logUx_norm, 'Ux_mean': Ux_m_s, 'Ux_std': Ux_s_s,
        'logUy_norm': logUy_norm, 'Uy_mean': Uy_m_s, 'Uy_std': Uy_s_s,
        'logDx_norm': logDx_norm, 'Dx_mean': Dx_m_s, 'Dx_std': Dx_s_s,
        'logDy_norm': logDy_norm, 'Dy_mean': Dy_m_s, 'Dy_std': Dy_s_s,
        'logVbx_norm': logVbx_norm, 'Vbx_mean': Vbx_m_s, 'Vbx_std': Vbx_s_s,
        'logVby_norm': logVby_norm, 'Vby_mean': Vby_m_s, 'Vby_std': Vby_s_s,
    }


def make_dataloaders(idx_tr, idx_val, idx_test, scalers):
    N_tr    = N_pisos_arr[idx_tr]
    counts  = np.bincount(N_tr, minlength=N_MAX_PISOS+1).astype(float)
    counts[counts == 0] = 1.0
    weights = 1.0 / counts[N_tr]
    sampler = WeightedRandomSampler(weights, num_samples=len(idx_tr), replacement=True)

    dl_tr  = DataLoader(ModalDataset(idx_tr,   scalers), batch_size=BATCH_SIZE,
                        sampler=sampler, drop_last=True)
    dl_val = DataLoader(ModalDataset(idx_val,  scalers), batch_size=BATCH_SIZE, shuffle=False)
    dl_tst = DataLoader(ModalDataset(idx_test, scalers), batch_size=BATCH_SIZE, shuffle=False)
    return dl_tr, dl_val, dl_tst


def build_model():
    return PINNModal_v4(N_INPUTS, N_MODOS, N_MAX_PISOS,
                        hidden=256, hidden_T1=128).to(device)


print('CELDA 6 OK — v16 ratio + scaler_pisos corregido')
print('  scaler_pisos: sobre pisos activos (corrección bug)')
print('  + ratio_dx_dy = drift_x / clip(drift_y, 1e-6)')

CELDA 6 OK — v16 ratio + scaler_pisos corregido
  scaler_pisos: sobre pisos activos (corrección bug)
  + ratio_dx_dy = drift_x / clip(drift_y, 1e-6)


## 📌 Celda 7 — Curriculum de entrenamiento v20 (definitivo)

### 🎯 ¿Qué hace esta celda?

Define el **currículo de aprendizaje** (Bengio et al., 2009) que **activa progresivamente** los términos de la pérdida a lo largo del entrenamiento, mediante la función `get_lambdas_and_active(epoch)`. Empieza solo con términos data-driven y va incorporando los físicos (φ, eigen, ortogonalidad) por etapas.

### 🗂️ Etapas (Fase 1: `EP_TOTAL = 600` épocas)

| Épocas | Etapa | Activa además de los data |
|---|---|---|
| 0 – 149 | E0 (frozen) | solo `data_resp`, `data_T_rest`, `ratio`; **encoder_T1 congelado** |
| 150 – 199 | S1 | + `data_T1`, + `phi` ($\lambda = 0.1$) |
| 200 – 349 | S2 | + `T1`, + `eig` ($\lambda = 0.1$) |
| 350 – 499 | S3 | + `ortho` ($\lambda = 10^{-5}$) |
| 500 – 599 | S4 | igual que S3 (consolidación) |

### 🧮 Pesos $\lambda$ por etapa

En todas las etapas: `data_resp = 2.0`, `data_T_rest = 1.0`, `ratio = 1.0`. Los términos `data_T1`, `T1`, `phi`, `eig` y `ortho` se encienden según la tabla anterior. Las banderas `stage_active` (`phi`, `eig`, `ortho`) controlan qué términos físicos se computan en cada época.

> **Nota:** durante las primeras 150 épocas se desactivan los gradientes del `encoder_T1` (entrenamiento en dos velocidades). El optimizador usa `CosineAnnealingWarmRestarts` (Loshchilov & Hutter, 2017).

### ✅ Lo que se obtiene

- Función `get_lambdas_and_active(epoch)` que devuelve el diccionario de pesos $\lambda$ y los flags `stage_active` para esa época.
- Constante `EP_TOTAL = 600` (Fase 1). La Fase 2 puede usar un horizonte distinto.

In [11]:
# CELDA 7 -- CURRICULUM v20
EP_TOTAL = 600

def get_lambdas_and_active(epoch):
    if epoch < 150:
        return (
            {'data_resp':2.0,'data_T_rest':1.0,'data_T1':0.0,
             'T1':0.0,'phi':0.0,'eig':0.0,'ortho':0.0,'ratio':1.0},
            {'phi':False,'eig':False,'ortho':False}
        )
    elif epoch < 200:
        return (
            {'data_resp':2.0,'data_T_rest':1.0,'data_T1':1.0,
             'T1':0.0,'phi':0.1,'eig':0.0,'ortho':0.0,'ratio':1.0},
            {'phi':True,'eig':False,'ortho':False}
        )
    elif epoch < 350:
        return (
            {'data_resp':2.0,'data_T_rest':1.0,'data_T1':1.0,
             'T1':1.0,'phi':0.1,'eig':0.1,'ortho':0.0,'ratio':1.0},
            {'phi':True,'eig':True,'ortho':False}
        )
    elif epoch < 500:
        return (
            {'data_resp':2.0,'data_T_rest':1.0,'data_T1':1.0,
             'T1':1.0,'phi':0.1,'eig':0.1,'ortho':1e-5,'ratio':1.0},
            {'phi':True,'eig':True,'ortho':True}
        )
    else:
        return (
            {'data_resp':2.0,'data_T_rest':1.0,'data_T1':1.0,
             'T1':1.0,'phi':0.1,'eig':0.1,'ortho':1e-5,'ratio':1.0},
            {'phi':True,'eig':True,'ortho':True}
        )

STAGES = None
print(f'CELDA 7 OK -- Curriculum v20')
print(f'  EP_TOTAL : {EP_TOTAL}')

CELDA 7 OK -- Curriculum v20
  EP_TOTAL : 600


## 📌 Celda 8 — Funciones de entrenamiento y métricas OOD (v20)

### 🎯 ¿Qué hace esta celda?

Define **todas las funciones núcleo** del experimento:
1. Análisis de los dominios A vs B.
2. Verificación de **diversidad** entre conjuntos.
3. Bucle de entrenamiento con staged training.
4. Cálculo de **métricas OOD** finales.

### 🔍 1. `tabla_dominios(trial_def, idx_pool_A, idx_pool_B)`

Para cada feature, calcula la **distancia normalizada** entre las medias de Pool A y Pool B:

$$
\Delta_{\text{norm}} = \frac{|\bar{x}_A - \bar{x}_B|}{\max(\sigma_A, \sigma_B, \epsilon)}
$$

Si $\Delta_{\text{norm}} > 0.3\sigma$, la feature **difiere significativamente** entre dominios. También calcula valores exclusivos (categorías presentes solo en A o solo en B) y estadísticas de la respuesta sísmica ($T_1$, $V_{b,x}$).

### 📊 2. `verificar_diversidad(idx_A1, idx_A2, idx_pool_B)`

Calcula **distancias mínimas euclidianas** en el espacio normalizado de features:

$$
d_{\min}(\mathcal{S}) = \min_{i \neq j \in \mathcal{S}} \|\tilde{x}_i - \tilde{x}_j\|_2
$$

- $d_{\min}^{A_1}$: diversidad dentro de A1 (debería ser > 1.0).
- $d_{\min}^{A \leftrightarrow B}$: **separación** entre A y B (debería ser > 1.0).

### 🏋️ 3. `train_trial(model, dl_tr, dl_val, scalers, trial_dir, ep_total, verbose)`

Loop de entrenamiento:

```
para cada época ep en [0, ep_total):
    1. Obtener lambdas y stages activos de get_lambdas_and_active(ep)
    2. Congelar/descongelar encoder_T1 según ep ⋚ 150
    3. TRAIN:
         ▸ forward → out = model(X, mask)
         ▸ loss, ld = compute_loss(batch, out, lambdas, scalers, active)
         ▸ backward + clip_grad_norm(max_norm=1.0)
         ▸ optimizer.step()
    4. VAL:
         ▸ forward sin gradientes
         ▸ acumular val_losses
         ▸ val_total = Σ λ_k * val_loss_k
    5. scheduler.step()  (Cosine warm restarts)
    6. Si val_total < best Y ep ≥ 250:
         ▸ guardar best_model.pt
```

> **Clip de gradientes**: `clip_grad_norm_(parameters, max_norm=1.0)` evita explosiones cuando los términos físicos se activan de golpe.

### 📏 4. `compute_metrics_ood(model, dl_ood, scalers)`

Calcula sobre el conjunto **fuera de distribución** (OOD = Pool B test):

#### 4.1 MAC promedio sobre 3 direcciones
$$
\text{MAC}_{\text{avg}} = \frac{1}{3}\left(\text{MAC}_{Y_1} + \text{MAC}_{T_1} + \text{MAC}_{X_1}\right)
$$
donde el slot 0 es Y, slot 1 es T (torsional), slot 2 es X.

#### 4.2 MAPE del periodo fundamental
$$
\text{T1\_MAPE} = 100 \cdot \mathbb{E}\!\left[\frac{|T_1^{\text{pred}} - T_1^{\text{true}}|}{T_1^{\text{true}} + \epsilon}\right]
$$

#### 4.3 MAPE del corte basal (promedio X-Y)
$$
\text{Vb\_MAPE} = \tfrac{1}{2}\big(\text{Vbx\_MAPE} + \text{Vby\_MAPE}\big)
$$

#### 4.4 MAPE de la deriva máxima
$$
\delta_{\max} = \max_{i, d} \delta_{d,i}, \qquad d \in \{x,y\}, \; i \in \{1,...,N_p\}
$$

> **Desnormalización**: se aplica $T = \exp(\widetilde{\log T} \cdot \sigma + \mu)$ antes de calcular MAPE.

### ✅ Lo que se obtiene

- 4 funciones globales: `tabla_dominios`, `verificar_diversidad`, `train_trial`, `compute_metrics_ood`.
- Patrón fijo de direcciones modales: `DIR_MODO = [Y, T, X, Y, T, X, ...]`.

In [12]:
# CELDA 8 -- FUNCIONES TRAIN Y METRICAS 

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

_scaler_div = StandardScaler()
_X_norm_div = _scaler_div.fit_transform(X_clean)

DIR_MODO = []
for r in range(18):
    ciclo = r % 3
    if ciclo == 0:   DIR_MODO.append('Y')
    elif ciclo == 1: DIR_MODO.append('T')
    else:            DIR_MODO.append('X')


def tabla_dominios(trial_def, idx_pool_A, idx_pool_B):
    rng_v  = np.random.default_rng(42)
    n_mues = min(500, len(idx_pool_A), len(idx_pool_B))
    idx_Am = rng_v.choice(idx_pool_A, n_mues, replace=False)
    idx_Bm = rng_v.choice(idx_pool_B, n_mues, replace=False)

    print(f"\n{'='*65}")
    print(f"TRIAL {trial_def['id']:02d} — {trial_def['nombre']}")
    print(f"  Pool A (TRAIN): {len(idx_pool_A)} casos")
    print(f"  Pool B (OOD)  : {len(idx_pool_B)} casos")

    difs = []
    for col in COLS_BASE:
        idx_c  = COLS_BASE.index(col)
        vA     = X_clean[idx_Am, idx_c]
        vB     = X_clean[idx_Bm, idx_c]
        sigma  = max(vA.std(), vB.std(), 1e-8)
        d_norm = abs(vA.mean() - vB.mean()) / sigma
        if d_norm > 0.3:
            difs.append((col, vA, vB, d_norm))
    difs.sort(key=lambda x: x[3], reverse=True)

    print(f"\n  FEATURES QUE DIFIEREN ENTRE DOMINIOS (Δ > 0.3σ):")
    print(f"  De {len(COLS_BASE)} features: {len(difs)} difieren significativamente")
    if difs:
        print(f"  {'Feature':<22} {'Pool A':>28} {'Pool B':>28} {'Δ':>6}")
        print(f"  {'-'*88}")
        for col, vA, vB, d in difs:
            signo  = '↑' if vB.mean() > vA.mean() else '↓'
            u_A    = sorted(set(np.round(vA, 2).tolist()))
            u_B    = sorted(set(np.round(vB, 2).tolist()))
            str_A  = str([round(v,2) for v in u_A]) if len(u_A)<=8 \
                     else f"[{vA.min():.2f}→{vA.max():.2f}] media={vA.mean():.2f}"
            str_B  = str([round(v,2) for v in u_B]) if len(u_B)<=8 \
                     else f"[{vB.min():.2f}→{vB.max():.2f}] media={vB.mean():.2f}"
            print(f"  {col:<22} {str_A:>28} {str_B:>28}  {signo}{d:.2f}σ")

    print(f"\n  RESPUESTA SÍSMICA:")
    for arr, nombre in [(T_r[:,0],'T1 [s]'), (Vb_x,'Vbx [kN]')]:
        vA = arr[idx_Am]; vB = arr[idx_Bm]
        print(f"  {nombre:<12} Pool A: [{vA.min():.2f}→{vA.max():.2f}] "
              f"media={vA.mean():.2f}   "
              f"Pool B: [{vB.min():.2f}→{vB.max():.2f}] "
              f"media={vB.mean():.2f}")
    print(f"{'='*65}")


def verificar_diversidad(idx_A1, idx_A2, idx_pool_B):
    rng_v = np.random.default_rng(42)
    n_m   = 300
    mA1   = rng_v.choice(idx_A1,     min(n_m, len(idx_A1)),     replace=False)
    mA2   = rng_v.choice(idx_A2,     min(n_m, len(idx_A2)),     replace=False)
    mB    = rng_v.choice(idx_pool_B, min(n_m, len(idx_pool_B)), replace=False)

    def min_dist_interna(idx_m):
        D = pairwise_distances(_X_norm_div[idx_m])
        np.fill_diagonal(D, np.inf)
        return D[D != np.inf].min()

    min_A1 = min_dist_interna(mA1)
    min_A2 = min_dist_interna(mA2)
    min_B  = min_dist_interna(mB)
    min_AB = pairwise_distances(_X_norm_div[mA1], _X_norm_div[mB]).min()

    def est(v): return '✅' if v > 1.0 else '⚠️'

    print(f"\n  DIVERSIDAD (dist_min espacio normalizado, meta > 1.0):")
    print(f"  A1={min_A1:.3f}{est(min_A1)}  "
          f"A2={min_A2:.3f}{est(min_A2)}  "
          f"B={min_B:.3f}{est(min_B)}  "
          f"| Sep A↔B={min_AB:.3f}{est(min_AB)}")

    return {
        'min_A1': round(min_A1, 4),
        'min_A2': round(min_A2, 4),
        'min_B' : round(min_B,  4),
        'min_AB': round(min_AB, 4),
    }


def train_trial(model, dl_tr, dl_val, scalers, trial_dir,
                ep_total=600, verbose=True):
    trial_dir.mkdir(parents=True, exist_ok=True)
    history         = []
    best_path       = trial_dir / 'best_model.pt'
    best_val_global = float('inf')
    best_epoch      = 0

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = CosineAnnealingWarmRestarts(
        optimizer, T_0=200, T_mult=2, eta_min=1e-6)

    params_T1 = (list(model.encoder_T1.parameters()) +
                 list(model.head_T1.parameters()))

    etapa_actual = -1
    stage_actual = -1

    for ep in range(ep_total):
        lambdas, active = get_lambdas_and_active(ep)

        if ep < 150:
            if etapa_actual != 0:
                etapa_actual = 0
                for p in params_T1:
                    p.requires_grad = False
                if verbose:
                    print(f'\n  ── Etapa 0 (ep {ep}) encoder_T1 CONGELADO ──')
        else:
            if etapa_actual != 1:
                etapa_actual = 1
                for p in params_T1:
                    p.requires_grad = True
                if verbose:
                    print(f'\n  ── Etapa 1 (ep {ep}) encoder_T1 DESCONGELADO ──')

        stage_nuevo = {150:1, 200:2, 350:3, 500:4}.get(ep, stage_actual)
        if stage_nuevo != stage_actual and ep >= 150:
            stage_actual = stage_nuevo
            if verbose:
                print(f'\n  ── Stage {stage_actual} (ep {ep}) '
                      f'T1λ={lambdas["T1"]} '
                      f'eig={lambdas["eig"]} '
                      f'lr={optimizer.param_groups[0]["lr"]:.2e} ──')

        model.train()
        tr_losses = {}
        for batch in dl_tr:
            optimizer.zero_grad()
            out  = model(batch['X'].to(device), batch['mask'].to(device))
            loss, ld = compute_loss(batch, out, lambdas, scalers, active)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            for k, v in ld.items():
                tr_losses[k] = tr_losses.get(k, 0) + v

        model.eval()
        val_losses = {}
        with torch.no_grad():
            for batch in dl_val:
                out  = model(batch['X'].to(device), batch['mask'].to(device))
                _, ld = compute_loss(batch, out, lambdas, scalers, active)
                for k, v in ld.items():
                    val_losses[k] = val_losses.get(k, 0) + v

        val_total = sum(lambdas.get(k, 0) * v
                        for k, v in val_losses.items())
        scheduler.step()

        if val_total < best_val_global and ep >= 250:
            best_val_global = val_total
            torch.save(model.state_dict(), best_path)
            best_epoch = ep

        row = {'etapa': etapa_actual, 'stage': stage_actual,
               'epoch': ep, 'val_total': val_total,
               'lr': optimizer.param_groups[0]['lr']}
        row.update({f'tr_{k}': v/max(len(dl_tr), 1)
                    for k, v in tr_losses.items()})
        row.update({f'val_{k}': v/max(len(dl_val), 1)
                    for k, v in val_losses.items()})
        history.append(row)

        if verbose and ep % 25 == 0:
            frozen_str = 'T1_frozen' if ep < 150 else 'T1_free'
            print(f'  ep{ep:4d}/{ep_total} '
                  f'val={val_total:.5f} '
                  f'best={best_val_global:.5f} (ep{best_epoch}) '
                  f'lr={optimizer.param_groups[0]["lr"]:.2e} '
                  f'[{frozen_str}]')

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(trial_dir / 'history.csv', index=False)
    return hist_df, best_val_global, best_epoch


def compute_metrics_ood(model, dl_ood, scalers):
    model.eval()

    logT_std_t  = torch.tensor(scalers['logT_std'],  dtype=torch.float32, device=device)
    logT_mean_t = torch.tensor(scalers['logT_mean'], dtype=torch.float32, device=device)
    Vbx_std_t   = torch.tensor([scalers['Vbx_std']], dtype=torch.float32, device=device)
    Vbx_mean_t  = torch.tensor([scalers['Vbx_mean']],dtype=torch.float32, device=device)
    Vby_std_t   = torch.tensor([scalers['Vby_std']], dtype=torch.float32, device=device)
    Vby_mean_t  = torch.tensor([scalers['Vby_mean']],dtype=torch.float32, device=device)
    Dx_std_t    = torch.tensor([scalers['Dx_std']],  dtype=torch.float32, device=device)
    Dx_mean_t   = torch.tensor([scalers['Dx_mean']], dtype=torch.float32, device=device)
    Dy_std_t    = torch.tensor([scalers['Dy_std']],  dtype=torch.float32, device=device)
    Dy_mean_t   = torch.tensor([scalers['Dy_mean']], dtype=torch.float32, device=device)

    MAC_Y1_list=[]; MAC_T1_list=[]; MAC_X1_list=[]
    T1_pred_list=[]; T1_true_list=[]
    Vbx_pred_list=[]; Vbx_true_list=[]
    Vby_pred_list=[]; Vby_true_list=[]
    drift_max_pred_list=[]; drift_max_true_list=[]

    with torch.no_grad():
        for batch in dl_ood:
            X_b    = batch['X'].to(device)
            mask_b = batch['mask'].to(device)

            logT_p, Phi_x_p, Phi_y_p, Phi_t_p, \
                Ux_p, Uy_p, dx_p, dy_p, Vb_p, ratio_p = model(X_b, mask_b)   # ← 10 outputs

            for slot, MAC_list, Phi_p, key in [
                (0, MAC_Y1_list, Phi_y_p, 'Phi_y'),
                (1, MAC_T1_list, Phi_t_p, 'Phi_theta'),
                (2, MAC_X1_list, Phi_x_p, 'Phi_x'),
            ]:
                phi_pred = Phi_p[:, :, slot] * mask_b
                phi_true = batch[key][:, :, slot].to(device) * mask_b
                num   = (phi_pred * phi_true).sum(dim=1) ** 2
                den_p = (phi_pred ** 2).sum(dim=1)
                den_t = (phi_true ** 2).sum(dim=1)
                den   = (den_p * den_t).clamp(min=1e-12)
                valido= (den_t > 1e-10)
                mac_r = torch.where(valido, num/den, torch.zeros_like(num))
                MAC_list.extend(mac_r[valido].cpu().numpy().tolist())

            T_pred  = torch.exp(logT_p * logT_std_t + logT_mean_t)
            T_true  = batch['T_r'].to(device)
            mask_T1 = T_true[:, 0] > 1e-6
            if mask_T1.sum() > 0:
                T1_pred_list.extend(T_pred[mask_T1, 0].cpu().numpy().tolist())
                T1_true_list.extend(T_true[mask_T1, 0].cpu().numpy().tolist())

            Vbx_pred = torch.exp(Vb_p[:,0]*Vbx_std_t + Vbx_mean_t)
            Vby_pred = torch.exp(Vb_p[:,1]*Vby_std_t + Vby_mean_t)
            Vbx_pred_list.extend(Vbx_pred.squeeze().cpu().numpy().tolist())
            Vbx_true_list.extend(batch['Vbx_raw'].cpu().numpy().tolist())
            Vby_pred_list.extend(Vby_pred.squeeze().cpu().numpy().tolist())
            Vby_true_list.extend(batch['Vby_raw'].cpu().numpy().tolist())

            # ← dy físico desde su scaler; dx = ratio × dy en escala física
            dy_real = torch.exp(dy_p*Dy_std_t+Dy_mean_t).clamp(min=1e-10) * mask_b   # ←
            dx_real = (ratio_p * dy_real) * mask_b                                   # ←
            drift_max_pred = torch.maximum(dx_real, dy_real).max(dim=1).values

            drift_x_raw    = batch['drift_x_raw'].to(device) * mask_b
            drift_y_raw    = batch['drift_y_raw'].to(device) * mask_b
            drift_max_true = torch.maximum(drift_x_raw, drift_y_raw).max(dim=1).values
            mask_drift     = drift_max_true > 1e-8
            if mask_drift.sum() > 0:
                drift_max_pred_list.extend(drift_max_pred[mask_drift].cpu().numpy().tolist())
                drift_max_true_list.extend(drift_max_true[mask_drift].cpu().numpy().tolist())

    MAC_Y1  = float(np.mean(MAC_Y1_list))  if MAC_Y1_list  else 0.0
    MAC_T1  = float(np.mean(MAC_T1_list))  if MAC_T1_list  else 0.0
    MAC_X1  = float(np.mean(MAC_X1_list))  if MAC_X1_list  else 0.0
    MAC_avg = (MAC_Y1 + MAC_T1 + MAC_X1) / 3.0

    T1_pred_a = np.array(T1_pred_list); T1_true_a = np.array(T1_true_list)
    T1_MAPE   = float(np.mean(np.abs(T1_pred_a-T1_true_a)/(T1_true_a+1e-8))*100)

    Vbx_pred_a = np.array(Vbx_pred_list); Vbx_true_a = np.array(Vbx_true_list)
    Vby_pred_a = np.array(Vby_pred_list); Vby_true_a = np.array(Vby_true_list)
    Vbx_MAPE   = float(np.mean(np.abs(Vbx_pred_a-Vbx_true_a)/(Vbx_true_a+1e-8))*100)
    Vby_MAPE   = float(np.mean(np.abs(Vby_pred_a-Vby_true_a)/(Vby_true_a+1e-8))*100)
    Vb_MAPE    = (Vbx_MAPE + Vby_MAPE) / 2.0

    drift_pred_a = np.array(drift_max_pred_list)
    drift_true_a = np.array(drift_max_true_list)
    drift_MAPE   = float(np.mean(
        np.abs(drift_pred_a-drift_true_a)/(drift_true_a+1e-8))*100)

    return {
        'MAC_avg'    : round(MAC_avg,   6),
        'MAC_Y1'     : round(MAC_Y1,    6),
        'MAC_T1'     : round(MAC_T1,    6),
        'MAC_X1'     : round(MAC_X1,    6),
        'T1_MAPE'    : round(T1_MAPE,   4),
        'Vb_MAPE'    : round(Vb_MAPE,   4),
        'Vbx_MAPE'   : round(Vbx_MAPE,  4),
        'Vby_MAPE'   : round(Vby_MAPE,  4),
        'drift_MAPE' : round(drift_MAPE, 4),
        'n_casos_ood': len(T1_pred_list),
    }


print('CELDA 8 OK — v20')

CELDA 8 OK — v20


## 📌 Celda 9 — Definición de los 31 trials OOD

### 🎯 ¿Qué hace esta celda?

Define los **31 escenarios experimentales** que se evaluarán. Cada trial divide el dataset en dos *pools* (A y B) según un criterio físico/estructural, permitiendo medir cómo **generaliza el PINN ante un *distributional shift*** (desplazamiento de distribución).

### 🗂️ Estructura de cada trial

```python
{
    'id'    : int,                 # identificador del trial
    'nombre': str,                  # descripción corta
    'mask_A': boolean array (N_cases,),   # define Pool A (TRAIN)
    'mask_B': boolean array (N_cases,),   # define Pool B (OOD)
    'tipo'  : 'conceptual' | 'complemento' | 'aleatorio_puro'
}
```

### 📚 Los 6 grupos de trials

| Grupo | Trials | Eje de distributional shift |
|---|---|---|
| **G1 — Nº de pisos** | T01–T05 | Edificios bajos vs altos |
| **G2 — Material** | T06–T10 | $f_c$, espesor de muros, $g_k$, tipo de suelo |
| **G3 — Geometría** | T11–T15 | Dimensiones de planta y núcleo |
| **G4 — Sísmica** | T16–T20 | Zona sísmica, tipo de suelo NCh 433 |
| **G5 — Tipología** | T21–T25 | Combinaciones de tipos de edificio |
| **G6 — Combinados** | T26–T29 | Mezclas mixtas |
| **Línea base** | T30, T31 | Aleatorio vs complemento, aleatorio puro |

### 🎯 Trials destacados (ejemplos)

| ID | Nombre | Pool A | Pool B |
|---|---|---|---|
| T02 | `N_bajo_vs_alto` | $N_p \leq 10$ | $N_p \geq 13$ |
| T10 | `suelo_AB_vs_CD` | Suelos rígidos (A,B) | Suelos blandos (C,D) |
| T16 | `zona_1_3_vs_zona_2` | Zonas 1 y 3 | Zona 2 |
| T20 | `fc_25_30_vs_fc_35_40` | $f_c \in \{25, 30\}$ MPa | $f_c \in \{35, 40\}$ MPa |
| T31 | **`aleatorio_puro`** | 5000 al azar | 5000 al azar | ← **línea base** sin shift |

### 🏛️ Justificación ingenieril (norma NCh 433)

- **Trial 10 (suelo AB vs CD)**: el tipo de suelo afecta $S_a(T)$ — espectro de aceleración. Suelos blandos (C, D) amplifican períodos largos.
- **Trial 16 (zona 1_3 vs 2)**: la zona sísmica determina $A_0$ (aceleración máxima efectiva). Zona 2 cubre la mayor parte de Chile central.
- **Trial 20 (fc bajo vs alto)**: $E = 4700\sqrt{f_c}$ → cambia rigidez $\mathbf{K}$ → cambia $T_r$.

### 🔄 Trials de reemplazo (v3)

Los trials originales T05, T10, T20, T26, T28, T29 tenían pools insuficientes (< 1600 casos). Se reemplazaron por criterios ingenierilmente significativos manteniendo factibilidad.

### ✅ Verificación de factibilidad

Cada trial debe cumplir:
$$
|\text{Pool A}| \geq 2N_{\text{train}} = 2000, \qquad |\text{Pool B}| \geq K_{\text{reps}}(N_{\text{val}} + N_{\text{test}}) = 1600
$$

Si no, se reducen las repeticiones a $K_{\text{reps}} = 2$.

### ✅ Lo que se obtiene

- Lista `TRIALS_DEF` con los 31 diccionarios.
- Tabla impresa de factibilidad (Pool A, Pool B, ¿ok para 4 reps?).

In [13]:
# CELDA 9 -- DEFINICION DE LOS 31 TRIALS OOD
# v3: reemplazo de 6 trials con pools insuficientes

N_p  = N_pisos_arr
fc   = X_clean[:, COLS_BASE.index('fc_MPa')]
h_s  = X_clean[:, COLS_BASE.index('h_story_m')]
n_u  = X_clean[:, COLS_BASE.index('n_unid_lado')]
L_m  = X_clean[:, COLS_BASE.index('L_mod_m')]
t_n  = X_clean[:, COLS_BASE.index('t_muro_nucleo_m')]
t_b  = X_clean[:, COLS_BASE.index('t_muro_borde_m')]
t_m  = X_clean[:, COLS_BASE.index('t_muro_mid_m')]
gk   = X_clean[:, COLS_BASE.index('gk_kN_m2')]
Ln   = X_clean[:, COLS_BASE.index('L_nucleo_m')]
Bn   = X_clean[:, COLS_BASE.index('B_nucleo_m')]
pd_  = X_clean[:, COLS_BASE.index('prof_depto_m')]
ac   = X_clean[:, COLS_BASE.index('ancho_corredor_m')]
s_A  = X_clean[:, COLS_BASE.index('suelo_A')]
s_C  = X_clean[:, COLS_BASE.index('suelo_C')]
s_B  = X_clean[:, COLS_BASE.index('suelo_B')]
s_D  = X_clean[:, COLS_BASE.index('suelo_D')]
z_1  = X_clean[:, COLS_BASE.index('zona_1')]
z_2  = X_clean[:, COLS_BASE.index('zona_2')]
z_3  = X_clean[:, COLS_BASE.index('zona_3')]
B2   = X_clean[:, COLS_BASE.index('activar_B2')]

Ln_med = float(np.median(Ln)); Bn_med = float(np.median(Bn))
pd_med = float(np.median(pd_)); tn_med = float(np.median(t_n))
tb_med = float(np.median(t_b))

TRIALS_DEF = [
    # ── Grupo 1: N_pisos ─────────────────────────────────────────────────
    {'id':1,  'nombre':'N_par_vs_impar',
     'mask_A':N_p%2==0,             'mask_B':N_p%2!=0},
    {'id':2,  'nombre':'N_bajo_vs_alto',
     'mask_A':N_p<=10,              'mask_B':N_p>=13},
    {'id':3,  'nombre':'N_6a9_vs_14a18',
     'mask_A':N_p<=9,               'mask_B':N_p>=14},
    {'id':4,  'nombre':'N_6_7_8_vs_15_16_17_18',
     'mask_A':N_p<=8,               'mask_B':N_p>=15},
    # T05 REEMPLAZO: gk_60_65_vs_70_75
    # Original (N_par_bajo_vs_impar_alto) tenía Pool_B=2289 insuficiente
    # Nuevo: carga gravitacional baja vs alta — afecta masa y por tanto T y Vb
    {'id':5,  'nombre':'gk_60_65_vs_70_75',
     'mask_A':gk<=6.5,              'mask_B':gk>=7.0},
    # ── Grupo 2: Material ─────────────────────────────────────────────────
    {'id':6,  'nombre':'fc_25_35_vs_30_40',
     'mask_A':(fc==25)|(fc==35),    'mask_B':(fc==30)|(fc==40)},
    {'id':7,  'nombre':'t_nucleo_fino_vs_grueso',
     'mask_A':t_n<=tn_med,          'mask_B':t_n>tn_med},
    {'id':8,  'nombre':'t_borde_fino_vs_grueso',
     'mask_A':t_b<=tb_med,          'mask_B':t_b>tb_med},
    {'id':9,  'nombre':'gk_bajo_vs_alto',
     'mask_A':gk<=6.5,              'mask_B':gk>=7.0},
    # T10 REEMPLAZO: suelo_AB_vs_suelo_CD
    # Original (fc_bajo_N_bajo_vs_fc_alto_N_alto) tenía Pool_A=1888 insuficiente
    # Nuevo: suelos rígidos (A,B) vs blandos (C,D) — afecta Sa(T) en NCh433
    {'id':10, 'nombre':'suelo_AB_vs_suelo_CD',
     'mask_A':(s_A==1)|(s_B==1),    'mask_B':(s_C==1)|(s_D==1)},
    # ── Grupo 3: Geometría ────────────────────────────────────────────────
    {'id':11, 'nombre':'n_unid_bajo_vs_alto',
     'mask_A':n_u<=3,               'mask_B':n_u>=5},
    {'id':12, 'nombre':'L_mod_bajo_vs_alto',
     'mask_A':L_m<=3.4,             'mask_B':L_m>=3.6},
    {'id':13, 'nombre':'L_nucleo_corto_vs_largo',
     'mask_A':Ln<Ln_med,            'mask_B':Ln>Ln_med},
    {'id':14, 'nombre':'B_nucleo_pequeno_vs_grande',
     'mask_A':Bn<Bn_med,            'mask_B':Bn>Bn_med},
    {'id':15, 'nombre':'prof_depto_bajo_vs_alto',
     'mask_A':pd_<pd_med,           'mask_B':pd_>pd_med},
    # ── Grupo 4: Sísmica ──────────────────────────────────────────────────
    {'id':16, 'nombre':'zona_1_3_vs_zona_2',
     'mask_A':(z_1==1)|(z_3==1),    'mask_B':z_2==1},
    {'id':17, 'nombre':'suelo_AC_vs_BD',
     'mask_A':(s_A==1)|(s_C==1),    'mask_B':(s_B==1)|(s_D==1)},
    {'id':18, 'nombre':'t_nucleo_min_vs_max',
     'mask_A':t_n<=np.percentile(t_n,25), 'mask_B':t_n>=np.percentile(t_n,75)},
    {'id':19, 'nombre':'zona_1_vs_zona_3',
     'mask_A':z_1==1,               'mask_B':z_3==1},
    # T20 REEMPLAZO: fc_25_30_vs_fc_35_40
    # Original (suelo_A_vs_suelo_D) tenía Pool_B=2503 insuficiente
    # Nuevo: fc bajo vs alto — afecta E=4700√fc y por tanto rigidez y periodos
    {'id':20, 'nombre':'fc_25_30_vs_fc_35_40',
     'mask_A':(fc==25)|(fc==30),    'mask_B':(fc==35)|(fc==40)},
    # ── Grupo 5: Tipología ────────────────────────────────────────────────
    {'id':21, 'nombre':'n_unid_pequeno_vs_grande',
     'mask_A':n_u<=2,               'mask_B':n_u>=4},
    {'id':22, 'nombre':'t_borde_min_vs_max',
     'mask_A':t_b<=np.percentile(t_b,25), 'mask_B':t_b>=np.percentile(t_b,75)},
    {'id':23, 'nombre':'h_bajo_vs_h_alto',
     'mask_A':h_s<=np.percentile(h_s,33), 'mask_B':h_s>=np.percentile(h_s,67)},
    {'id':24, 'nombre':'t_mid_min_vs_max',
     'mask_A':t_m<=np.percentile(t_m,25), 'mask_B':t_m>=np.percentile(t_m,75)},
    {'id':25, 'nombre':'corredor_angosto_vs_ancho',
     'mask_A':ac<=np.percentile(ac,33),   'mask_B':ac>=np.percentile(ac,67)},
    # ── Grupo 6: Combinados ───────────────────────────────────────────────
    # T26 REEMPLAZO: t_nucleo_018_020_vs_028_030
    # Original (N_par_fc_bajo_vs_N_impar_fc_alto) tenía Pool_A=2670 insuficiente
    # Nuevo: espesor núcleo delgado vs grueso — afecta rigidez torsional directamente
    {'id':26, 'nombre':'t_nucleo_018_020_vs_028_030',
     'mask_A':t_n<=0.20,            'mask_B':t_n>=0.28},
    {'id':27, 'nombre':'n_unid_2_3_vs_4_5',
     'mask_A':(n_u==2)|(n_u==3),    'mask_B':(n_u==4)|(n_u==5)},
    # T28 REEMPLAZO: corredor_angosto_vs_ancho (percentiles)
    # Original (L_mod_extremo_bajo_vs_alto) tenía Pool_B=3429 — casi suficiente
    # Nuevo: corredor afecta distribución de masa en Y — variable no explorada en extremos
    {'id':28, 'nombre':'corredor_angosto_p33_vs_ancho_p67',
     'mask_A':ac<=np.percentile(ac,33),   'mask_B':ac>=np.percentile(ac,67)},
    # T29 REEMPLAZO: fc_alto_vs_gk_alto
    # Original (h_story_min_vs_max) tenía Pool_B=2519 insuficiente
    # Nuevo: fc alto (alta rigidez) vs gk alto (alta masa) — separación mixta
    # que no existe en ningún otro trial — afecta ratio masa/rigidez
    {'id':29, 'nombre':'fc_alto_vs_gk_alto',
     'mask_A':fc>=35,               'mask_B':gk>=7.0},
    # ── Trial 30: Aleatorio vs complemento ───────────────────────────────
    {'id':30, 'nombre':'aleatorio_vs_complemento',
     'mask_A':None, 'mask_B':None, 'tipo':'complemento'},
    # ── Trial 31: Aleatorio puro — LÍNEA BASE ────────────────────────────
    {'id':31, 'nombre':'aleatorio_puro',
     'mask_A':None, 'mask_B':None, 'tipo':'aleatorio_puro'},
]

# ── Verificar factibilidad ────────────────────────────────────────────────
print(f"{'ID':<4} {'Nombre':<45} {'Pool_A':>7} {'Pool_B':>7} {'4_reps':>7}")
print("-"*72)

n_ok = 0
for t in TRIALS_DEF:
    tipo = t.get('tipo', 'conceptual')
    if tipo == 'complemento':
        n_A, n_B = 4000, 6000
    elif tipo == 'aleatorio_puro':
        n_A, n_B = 5000, 5000
    else:
        n_A = int(t['mask_A'].sum())
        n_B = int(t['mask_B'].sum())

    ok_4 = n_A >= MIN_POOL_A and n_B >= MIN_POOL_B
    flag = '✅' if ok_4 else '⚠️'
    if ok_4: n_ok += 1
    print(f"{t['id']:<4} {t['nombre']:<45} {n_A:>7} {n_B:>7}  {flag}")

print(f"\nTrials con 4 reps completas: {n_ok}/{len(TRIALS_DEF)}")
print(f"CELDA 9 OK — {len(TRIALS_DEF)} trials definidos")

ID   Nombre                                         Pool_A  Pool_B  4_reps
------------------------------------------------------------------------
1    N_par_vs_impar                                   5367    4633  ✅
2    N_bajo_vs_alto                                   3834    4637  ✅
3    N_6a9_vs_14a18                                   3100    3914  ✅
4    N_6_7_8_vs_15_16_17_18                           2318    3121  ✅
5    gk_60_65_vs_70_75                                4969    5031  ✅
6    fc_25_35_vs_30_40                                4989    5011  ✅
7    t_nucleo_fino_vs_grueso                          6650    3350  ✅
8    t_borde_fino_vs_grueso                           6005    3995  ✅
9    gk_bajo_vs_alto                                  4969    5031  ✅
10   suelo_AB_vs_suelo_CD                             4961    5039  ✅
11   n_unid_bajo_vs_alto                              4676    2981  ✅
12   L_mod_bajo_vs_alto                               4157    3438  ✅
13   L_nucle

In [14]:
# ============================================================
# CELDA 13 — FASE 2: REENTRENAMIENTO CON DATOS COMPLETOS
# Trial 16 — zona_1_3_vs_zona_2
# Arquitectura definitiva: PINNModal_v4b
# Loss definitiva:         compute_loss_v16
# Curriculum definitivo:   v20 staged training 600 épocas
# Dataset:                 v6 nuevo — B4-INF dividido por hall
# ============================================================

from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# DEFINITIVO_DIR y TRIALS_DIR vienen del panel (CELDA 0)

TRIAL_FASE2  = 16
NOMBRE_FASE2 = 'zona_1_3_vs_zona_2'
EP_FASE2     = 600
SEED_FASE2   = 2026

variables_necesarias = ['X_clean', 'N_pisos_arr', 'COLS_BASE',
                        'T_r', 'Vb_x', 'Vb_y', 'drift_x', 'drift_y',
                        'build_model', 'compute_scalers',
                        'make_dataloaders', 'compute_metrics_ood',
                        'compute_loss', 'TRIALS_DEF',
                        'N_MODOS', 'N_MODOS_PHYS', 'N_MAX_PISOS',
                        'get_lambdas_and_active']
faltantes = [v for v in variables_necesarias if v not in globals()]
if faltantes:
    print(f"⚠️  Faltan: {faltantes}")
    raise SystemExit

# ── Datos ──────────────────────────────────────────────────────────────────
trial_def  = next(t for t in TRIALS_DEF if t['id'] == TRIAL_FASE2)
idx_pool_A = np.where(trial_def['mask_A'])[0]
idx_pool_B = np.where(trial_def['mask_B'])[0]

rng_b       = np.random.default_rng(SEED_FASE2 + TRIAL_FASE2 * 200)
idx_B_shuf  = rng_b.permutation(idx_pool_B)
idx_val_f2  = idx_B_shuf[:400]
idx_test_f2 = idx_B_shuf[400:]

scalers_f2 = compute_scalers(idx_pool_A)
dl_tr_f2, dl_val_f2, dl_test_f2 = make_dataloaders(
    idx_pool_A, idx_val_f2, idx_test_f2, scalers_f2)

print(f"\n{'='*70}")
print(f"FASE 2 — Trial {TRIAL_FASE2:02d} — {NOMBRE_FASE2}")
print(f"{'='*70}")
print(f"  Pool A (TRAIN): {len(idx_pool_A)} casos")
print(f"  Pool B (OOD)  : {len(idx_pool_B)} casos")
print(f"  Val           : {len(idx_val_f2)} casos")
print(f"  Test          : {len(idx_test_f2)} casos")
print(f"  Épocas        : {EP_FASE2}")
print(f"  Arquitectura  : PINNModal_v4b")
print(f"  Loss          : compute_loss_v16")
print(f"  Curriculum    : v20 staged training")
print(f"  HDF5          : {H5_MODAL.name}")
print(f"  Guardado en   : {DEFINITIVO_DIR}")

# ── Entrenamiento ──────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"Iniciando entrenamiento Fase 2...")
print(f"{'─'*60}")

model_f2  = build_model()
best_path = DEFINITIVO_DIR / 'model_fase2_definitivo.pt'

optimizer = torch.optim.Adam(model_f2.parameters(), lr=1e-3)
scheduler = CosineAnnealingWarmRestarts(
    optimizer, T_0=200, T_mult=2, eta_min=1e-6)

params_T1 = (list(model_f2.encoder_T1.parameters()) +
             list(model_f2.head_T1.parameters()))

history_f2      = []
best_val_global = float('inf')
best_epoch_f2   = 0
etapa_actual    = -1
stage_actual    = -1
t_inicio        = datetime.now()

for ep in range(EP_FASE2):
    lambdas, active = get_lambdas_and_active(ep)

    # ── Congelar/descongelar encoder_T1 ──────────────────────────────────
    if ep < 150:
        if etapa_actual != 0:
            etapa_actual = 0
            for p in params_T1:
                p.requires_grad = False
            print(f"\n  ── Etapa 0 (ep {ep}) encoder_T1 CONGELADO ──")
    else:
        if etapa_actual != 1:
            etapa_actual = 1
            for p in params_T1:
                p.requires_grad = True
            print(f"\n  ── Etapa 1 (ep {ep}) encoder_T1 DESCONGELADO ──")

    stage_nuevo = {150:1, 200:2, 350:3, 500:4}.get(ep, stage_actual)
    if stage_nuevo != stage_actual and ep >= 150:
        stage_actual = stage_nuevo
        print(f"\n  ── Stage {stage_actual} (ep {ep}) "
              f"T1λ={lambdas['T1']} "
              f"eig={lambdas['eig']} "
              f"lr={optimizer.param_groups[0]['lr']:.2e} ──")

    # ── Train ─────────────────────────────────────────────────────────────
    model_f2.train()
    tr_losses = {}
    for batch in dl_tr_f2:
        optimizer.zero_grad()
        out  = model_f2(batch['X'].to(device),
                        batch['mask'].to(device))
        loss, ld = compute_loss(batch, out, lambdas, scalers_f2, active)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_f2.parameters(), 1.0)
        optimizer.step()
        for k, v in ld.items():
            tr_losses[k] = tr_losses.get(k, 0) + v

    # ── Val ───────────────────────────────────────────────────────────────
    model_f2.eval()
    val_losses = {}
    with torch.no_grad():
        for batch in dl_val_f2:
            out  = model_f2(batch['X'].to(device),
                            batch['mask'].to(device))
            _, ld = compute_loss(batch, out, lambdas, scalers_f2, active)
            for k, v in ld.items():
                val_losses[k] = val_losses.get(k, 0) + v

    val_total = sum(lambdas.get(k, 0) * v
                    for k, v in val_losses.items())
    scheduler.step()

    if val_total < best_val_global and ep >= 250:
        best_val_global = val_total
        torch.save(model_f2.state_dict(), best_path)
        best_epoch_f2 = ep

    row = {'etapa': etapa_actual, 'stage': stage_actual,
           'epoch': ep, 'val_total': val_total,
           'lr': optimizer.param_groups[0]['lr']}
    row.update({f'tr_{k}': v/max(len(dl_tr_f2),1)
                for k,v in tr_losses.items()})
    row.update({f'val_{k}': v/max(len(dl_val_f2),1)
                for k,v in val_losses.items()})
    history_f2.append(row)

    if ep % 25 == 0:
        frozen_str = 'T1_frozen' if ep < 150 else 'T1_free'
        print(f"  ep{ep:4d}/{EP_FASE2} "
              f"val={val_total:.5f} "
              f"best={best_val_global:.5f} (ep{best_epoch_f2}) "
              f"lr={optimizer.param_groups[0]['lr']:.2e} "
              f"[{frozen_str}]")

t_total = (datetime.now() - t_inicio).total_seconds() / 60
print(f"\n  Completado en {t_total:.1f} min | best_epoch={best_epoch_f2}")

pd.DataFrame(history_f2).to_csv(
    DEFINITIVO_DIR / 'history_fase2_definitivo.csv', index=False)

# ── Evaluación ────────────────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"Evaluando sobre Test OOD ({len(idx_test_f2)} casos)...")
print(f"{'─'*60}")

model_f2.load_state_dict(torch.load(best_path, map_location=device))
metrics_f2 = compute_metrics_ood(model_f2, dl_test_f2, scalers_f2)

with open(TRIALS_DIR / vfile('all_results','json'), encoding='utf-8') as f:
    r_f1 = json.load(f)[str(TRIAL_FASE2)]

print(f"\n{'='*70}")
print(f"COMPARACION FASE 1 vs FASE 2 — Trial {TRIAL_FASE2:02d} {NOMBRE_FASE2}")
print(f"{'='*70}")
print(f"  {'Metrica':<14} {'Fase 1':>12} {'Fase 2':>12} {'Mejora':>12}")
print(f"  {'-'*52}")
for k, label in [('MAC_avg',    'MAC_avg   '),
                 ('T1_MAPE',    'T1_MAPE % '),
                 ('Vb_MAPE',    'Vb_MAPE % '),
                 ('drift_MAPE', 'drift_MAPE%')]:
    v_f1 = r_f1['media'][k]
    v_f2 = metrics_f2[k]
    if k == 'MAC_avg':
        mejora = f"+{(v_f2-v_f1)*100:.3f}pp"
    else:
        pct    = ((v_f1 - v_f2) / v_f1) * 100
        mejora = f"{pct:+.1f}%{'↓' if pct>0 else '↑'}"
    print(f"  {label:<14} {v_f1:>12.4f} {v_f2:>12.4f} {mejora:>12}")

print(f"{'='*70}")
print(f"  Pool A Fase 1 : {r_f1['n_train']} casos")
print(f"  Pool A Fase 2 : {len(idx_pool_A)} casos")
print(f"  best_epoch    : {best_epoch_f2}")
print(f"{'='*70}")

# ── Guardar ───────────────────────────────────────────────────────────────
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):    return int(obj)
        if isinstance(obj, (np.floating,)):   return float(obj)
        if isinstance(obj, np.ndarray):       return obj.tolist()
        if isinstance(obj, (np.bool_, bool)): return bool(obj)
        return super().default(obj)

resultado_f2 = {
    'trial'        : TRIAL_FASE2,
    'nombre'       : NOMBRE_FASE2,
    'arquitectura' : 'PINNModal_v4b',
    'loss'         : 'compute_loss_v16',
    'curriculum'   : 'v20_staged_training',
    'dataset'      : 'v6 nuevo — B4-INF dividido por hall (20260523)',
    'h5_usado'     : H5_MODAL.name,
    'n_train_fase2': len(idx_pool_A),
    'n_test_fase2' : len(idx_test_f2),
    'ep_fase2'     : EP_FASE2,
    'best_epoch'   : best_epoch_f2,
    'tiempo_min'   : round(t_total, 2),
    'metricas_f2'  : metrics_f2,
    'metricas_f1'  : {k: r_f1['media'][k]
                      for k in ['MAC_avg','T1_MAPE',
                                'Vb_MAPE','drift_MAPE']},
    'referencias'  : [
        'Wang et al. (2022) — gradient pathologies PINNs',
        'Bengio et al. (2009) — Curriculum Learning',
        'Howard & Ruder (2018) — ULMFiT',
        'Loshchilov & Hutter (2017) — CosineAnnealingWarmRestarts',
        'Raissi et al. (2019) — Journal Comp. Physics 378',
        'NCh433 Mod.2012 + DS61',
    ],
}

with open(DEFINITIVO_DIR / 'resultado_fase2_definitivo.json',
          'w', encoding='utf-8') as f:
    json.dump(resultado_f2, f, indent=4,
              ensure_ascii=False, cls=NpEncoder)

print(f"\nArchivos guardados en: {DEFINITIVO_DIR}")
print(f"  model_fase2_definitivo.pt")
print(f"  history_fase2_definitivo.csv")
print(f"  resultado_fase2_definitivo.json")


FASE 2 — Trial 16 — zona_1_3_vs_zona_2
  Pool A (TRAIN): 6623 casos
  Pool B (OOD)  : 3377 casos
  Val           : 400 casos
  Test          : 2977 casos
  Épocas        : 600
  Arquitectura  : PINNModal_v4b
  Loss          : compute_loss_v16
  Curriculum    : v20 staged training
  HDF5          : dataset_familyB_OPENSEES_v6_DEFINITIVO.h5
  Guardado en   : C:\Users\rodri\Documents\PINN\b2_proyecto\models\trials_31\V8\DEFINITIVO_CORREGIDO

────────────────────────────────────────────────────────────
Iniciando entrenamiento Fase 2...
────────────────────────────────────────────────────────────

  ── Etapa 0 (ep 0) encoder_T1 CONGELADO ──
  ep   0/600 val=12.68128 best=inf (ep0) lr=1.00e-03 [T1_frozen]
  ep  25/600 val=9.66132 best=inf (ep0) lr=9.59e-04 [T1_frozen]
  ep  50/600 val=9.21425 best=inf (ep0) lr=8.48e-04 [T1_frozen]
  ep  75/600 val=9.25154 best=inf (ep0) lr=6.84e-04 [T1_frozen]
  ep 100/600 val=9.12417 best=inf (ep0) lr=4.93e-04 [T1_frozen]
  ep 125/600 val=9.15466 best=inf 

In [ ]:
# ============================================================
# CELDA 14 — COMPARACION LIMPIA: modelo VIEJO vs NUEVO
# AUTOCONTENIDA: reconstruye el split del Trial 16 y evalua los
# DOS modelos ella sola (no necesita CELDA 13 en memoria).
# Ambos vs el MISMO GT CORREGIDO y el MISMO test set, cada uno
# decodificado con SUS propios scalers.
# ============================================================
import pickle

TRIAL_FASE2 = 16
SEED_FASE2  = 2026

# ── Split del Trial 16 (identico a CELDA 13) ───────────────────────────────
trial_def  = next(t for t in TRIALS_DEF if t['id'] == TRIAL_FASE2)
idx_pool_A = np.where(trial_def['mask_A'])[0]
idx_pool_B = np.where(trial_def['mask_B'])[0]
rng_b      = np.random.default_rng(SEED_FASE2 + TRIAL_FASE2 * 200)
idx_B_shuf = rng_b.permutation(idx_pool_B)
idx_val_f2, idx_test_f2 = idx_B_shuf[:400], idx_B_shuf[400:]

# ── NUEVO modelo (reentrenado con datos corregidos) + sus scalers ──────────
scalers_new = compute_scalers(idx_pool_A)
_, _, dl_test_new = make_dataloaders(idx_pool_A, idx_val_f2, idx_test_f2, scalers_new)
model_new = build_model()
model_new.load_state_dict(torch.load(DEFINITIVO_DIR / 'model_fase2_definitivo.pt',
                                     map_location=device))
model_new.eval()
metrics_new = compute_metrics_ood(model_new, dl_test_new, scalers_new)

# ── VIEJO modelo (produccion, datos viejos) + sus scalers ──────────────────
OLD_DIR = TRIALS_DIR / 'DEFINITIVO'
with open(OLD_DIR / 'scalers_trial16.pkl', 'rb') as f:
    st = pickle.load(f)
scalers_old = dict(st)   # el pkl solo trae stats -> reconstruimos los *_norm
scalers_old['logT_norm']   = (logT_global - st['logT_mean']) / st['logT_std']
scalers_old['logUx_norm']  = (logUx  - st['Ux_mean'])  / st['Ux_std']
scalers_old['logUy_norm']  = (logUy  - st['Uy_mean'])  / st['Uy_std']
scalers_old['logDx_norm']  = (logDx  - st['Dx_mean'])  / st['Dx_std']
scalers_old['logDy_norm']  = (logDy  - st['Dy_mean'])  / st['Dy_std']
scalers_old['logVbx_norm'] = (logVbx - st['Vbx_mean']) / st['Vbx_std']
scalers_old['logVby_norm'] = (logVby - st['Vby_mean']) / st['Vby_std']

_, _, dl_test_old = make_dataloaders(idx_pool_A, idx_val_f2, idx_test_f2, scalers_old)
model_old = build_model()
model_old.load_state_dict(torch.load(OLD_DIR / 'model_fase2_definitivo.pt',
                                     map_location=device))
model_old.eval()
metrics_old = compute_metrics_ood(model_old, dl_test_old, scalers_old)

# ── Tabla ──────────────────────────────────────────────────────────────────
print(f"\n{'='*74}")
print(f"COMPARACION LIMPIA - ambos vs GT CORREGIDO  (test OOD {len(idx_test_f2)} casos)")
print(f"{'='*74}")
print(f"  {'Metrica':<14} {'VIEJO->corr':>14} {'NUEVO->corr':>14} {'Recupero':>14}")
print(f"  {'-'*56}")
for k, label in [('MAC_avg','MAC_avg   '),('T1_MAPE','T1_MAPE % '),
                 ('Vb_MAPE','Vb_MAPE % '),('drift_MAPE','drift_MAPE%')]:
    vo = metrics_old[k]; vn = metrics_new[k]
    rec = f"{(vn-vo)*100:+.3f}pp" if k=='MAC_avg' else f"{vo-vn:+.2f} pp"
    print(f"  {label:<14} {vo:>14.4f} {vn:>14.4f} {rec:>14}")
print(f"{'='*74}")
print("  VIEJO->corr : produccion (datos viejos), medido vs GT corregido")
print("  NUEVO->corr : reentrenado con datos corregidos, vs GT corregido")
print("  Recupero    : cuanto BAJO el error (pp) al reentrenar  (+ = mejor)")